# Sideband Analysis for the ANNIE NCQE cross section

This particular script samples the >2 sigma inter-bunch space sideband to constrain the external + skyshine rate. The script uses MC samples + full FY22-23 data (2.57e20 POT) used also for the cross section extraction.

--------------

### Setup, loading packages, and initial declarations

In [ ]:
print('Loading Packages...\n')
%matplotlib osx
import sys, os          
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import trange
import numpy as np
from scipy.optimize import curve_fit
import pandas as pd
from collections import defaultdict
import scipy.stats
import matplotlib.gridspec as gridspec
from scipy.stats import chi2_contingency
import random
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
import matplotlib.patches as mpatches
font = {'family' : 'serif', 'size' : 14 }
mpl.rc('font', **font)
mpl.rcParams['mathtext.fontset'] = 'cm' # Set the math font to Computer Modern
mpl.rcParams['legend.fontsize'] = 16
# Set default figure and axes face colors to white
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
    

# initialization ------------------------

# tank dimensions:
radius = 1.524
half_height = 1.98

# NCQE FV definition
FV_radius = 0.7
FV_half_height = 1.0


# # # # # # # # # # # # # 
# Bunch cutting

bunch_sigma = 3.59*2   # SIDEBAND 2sigma (we'll cut outside)

# define extent of the spill
bunch_time_cutoff = 1560    # data
time_to_prompt_end = 258    # MC approximate time from end of spill to 2us cutoff (chain last MC bunch to observed data last bunch)

# MC extracted CCinc bunch centers (https://github.com/S81D/BNB_bunches)
centers = np.array([30.55620074944292, 50.23596632813184, 68.03027546306257, 87.65149609251257, 106.38480577249247,
                    125.28116343989417, 144.32309666797107, 163.1397586239191, 181.97778394194054, 201.06274310271073,
                    220.16343853423476, 238.30288809166953, 258.38514881200354, 276.8146398611303, 296.0884356855441,
                    314.35016323142816, 333.72419652049575, 352.9437944077438, 371.260196759933, 390.08407553949974,
                    409.44913114396746, 427.94715055258985, 447.770917943728, 465.9339836950206, 484.7186644716273,
                    504.4287698808072, 523.0814143742606, 542.3141201678939, 561.2254420066273, 579.8111323601904,
                    597.9945900965694, 618.0700927506651, 636.5621477504429, 655.7874418315001, 674.4764151744993,
                    693.6905256979796, 712.4819567023824, 731.0362230741698, 750.2744038347329, 769.2102243656747,
                    788.2453849590231, 807.0011262101622, 825.9067358626738, 845.3047059731147, 864.0653260882543,
                    883.0323070032683, 901.6429155761502, 920.6808374077715, 939.6712240835648, 958.6990235603214,
                    977.72856819438, 996.2787338241341, 1015.08703056768, 1034.3375283354696, 1053.9759092673044,
                    1072.052984198524, 1091.129322973113, 1110.3389344568238, 1128.965140612168, 1148.3985550156349,
                    1166.982403872106, 1186.220604492118, 1204.4016122615021, 1223.770899436999, 1242.691800785879,
                    1261.9422162459668, 1280.3202276072477, 1299.702530449622, 1318.214516959161, 1337.6134384701843,
                    1356.5308849686603, 1375.0603202255497, 1393.91335999822, 1413.0147004160135, 1431.9961602521892,
                    1451.5391995383018, 1470.035773429933, 1488.7824718348304, 1507.6311809765648, 1526.7701548484445,
                    1545.2663160965799])

# spill width for data

# FY22
spill_start_22 = 206
spill_end_22 = 1739
centers_data_22 = np.array([215.7494731, 234.5402877, 252.6960547, 271.922027, 290.5568422,
                            309.4619782, 328.3685976, 347.1585617, 366.1694916, 384.6981959,
                            403.679617, 422.6118675, 441.7254035, 460.6980149, 479.8100942,
                            498.6283802, 517.4495414, 536.5487368, 555.6267941, 574.5979937,
                            593.0192376, 612.3368162, 630.8397525, 649.8131052, 669.2991329,
                            687.9304207, 706.7121484, 725.7727685, 744.8524787, 763.6331082,
                            782.3583396, 801.7439406, 820.2399223, 839.2074824, 858.2099129,
                            876.8574359, 896.0077987, 915.1144969, 934.2965022, 953.2212701,
                            971.751155, 990.5106562, 1009.956462, 1028.540103, 1047.63785,
                            1066.887371, 1085.900211, 1104.166114, 1123.524265, 1142.494186,
                            1161.263316, 1180.440215, 1199.068621, 1217.817441, 1236.616553,
                            1256.531327, 1274.887169, 1293.907079, 1313.039446, 1331.26644,
                            1350.901016, 1369.621886, 1388.363869, 1407.764603, 1425.900837,
                            1445.194635, 1464.29313, 1483.001877, 1502.552265, 1520.924771,
                            1539.732641, 1558.053482, 1577.955086, 1596.717582, 1616.106719,
                            1634.94341, 1653.694115, 1672.695675, 1691.851104, 1710.382291, 1729.90487])

# FY23
spill_start_23 = 189
spill_end_23 = 1719
centers_data_23 = np.array([198.3700379, 215.5394396, 234.2243011, 251.5523355, 272.0954178,
                            290.6241403, 308.5785171, 328.6072512, 347.7788392, 366.7606742,
                            385.8724421, 403.4237083, 422.6212047, 441.7513664, 460.8075196,
                            479.1954063, 499.1118531, 518.0524434, 536.2365549, 555.4753724,
                            574.4827687, 592.706705, 612.5075049, 631.5683442, 650.1400917,
                            669.2300559, 688.445265, 706.8814584, 725.5179244, 745.1280734,
                            763.1263233, 782.1646795, 801.9846119, 820.4792799, 839.8275515,
                            858.4967685, 876.8179439, 896.6128644, 915.5773794, 933.9912385,
                            953.7637369, 971.8604713, 992.8093626, 1009.494428, 1028.17604,
                            1047.085112, 1067.172794, 1084.712851, 1104.77286, 1123.612654,
                            1143.01596, 1161.339718, 1180.141252, 1200.09213, 1218.150494,
                            1237.658053, 1256.417415, 1274.927169, 1293.037241, 1312.298147,
                            1331.977857, 1351.217822, 1370.450652, 1388.800942, 1407.751471,
                            1426.765537, 1445.82194, 1464.015855, 1483.30137, 1502.409606,
                            1520.862787, 1539.567404, 1560.027696, 1577.842804, 1597.580137,
                            1616.224347, 1635.150248, 1654.172443, 1672.743926, 1691.044861, 1710.421766])



# # # # # # # # # # # # # 
# Helpful functions

# reconstructed energy
def reco_energy(pe):
    return (pe + 2.90)/5.90

# bunch cutting from extrapolated CCinc centers
def bunch_cutting(cluster_time, centers):
    avg_diff = 0.5081
    return np.any(np.abs( (centers - avg_diff) - cluster_time) < bunch_sigma)

# Charge coherence
def pairwise_relative_direction(hitX, hitY, hitZ, hitPE, hitT):
    
    hitX = np.array(hitX)
    hitY = np.array(hitY)
    hitZ = np.array(hitZ)
    hitPE = np.array(hitPE)
    hitT = np.array(hitT)

    # Combine and sort by time
    hits = list(zip(hitX, hitY, hitZ, hitPE, hitT))
    hits.sort(key=lambda h: h[4])  # sort by time
    
    mini = min(hitT)
    filtered_hits = []
    # Process remaining hits
    for hit in hits:
        x_i, y_i, z_i, pe_i, t_i = hit

        if (t_i - mini) > 13:
            continue
            
        filtered_hits.append(hit)

    # Extract filtered values
    filtX = np.array([h[0] for h in filtered_hits])
    filtY = np.array([h[1] for h in filtered_hits])
    filtZ = np.array([h[2] for h in filtered_hits])
    filtPE = np.array([h[3] for h in filtered_hits])
    
    n = len(filtX)
    if n < 2:
        return np.array([0.0, 0.0, 0.0])
    
    vec = np.zeros(3)
    for i in range(n):
        for j in range(i+1, n):
            dx = filtX[j] - filtX[i]
            dy = filtY[j] - filtY[i]
            dz = filtZ[j] - filtZ[i]
            r = np.sqrt(dx**2 + dy**2 + dz**2)
            if r == 0:
                continue
            
            # Direction between hits
            dvec = np.array([dx, dy, dz]) / r
            weight = filtPE[i] * filtPE[j]
            vec += weight * dvec
    
    norm = np.linalg.norm(vec)
    if norm == 0:
        return np.array([0.0, 0.0, 0.0])
    return vec / norm

# Parabolic charge coherence cut
def is_inside_rotated_parabola(Z_val, Y_val, a=-0.45, c=0.4, b=0.0, theta_deg=275):
    # Convert degrees to radians
    theta = np.deg2rad(theta_deg)
    
    # Inverse rotation (world → local)
    u =  Z_val*np.cos(theta) + Y_val*np.sin(theta)
    v = -Z_val*np.sin(theta) + Y_val*np.cos(theta)
    
    # Equation in local coords: v = a*u^2 + b*u + c
    # If opening is along +v, "inside" means v >= parabola
    return v < a*u**2 + b*u + c


print('done')

## I provide here two pathways:

### 1. Thesis analysis
These cells were the original machinery used to analyze the 2 sigma sideband region as outlined in Steven Doran's thesis; the skyshine rate was extracted and an additional dirt scaling factor was obtained to compare to the CCinc. It is important to note that this script utilizes GENIE v3.0.6 CV predictions with NO re-weighting (and hence no MicroBooNE tune). Later analyses (CCinc + the ultimate NCQE cross section calculation) have utilize the MicroBooNE tune. The reason for the discrepency is purely a "I forgot to apply the weights early on and wasn't using them, but for the actual cross section and especially the CCinc sideband I should use it, as it better scales the CC background". 

Keep in mind the skyshine rate was determined first by using the full spill window, then a later 1/3 spill cut was used to reduce the contribution from this background. If determining the skyshine rate from scratch, disable the "early" (25th) bunch cut.

Also it was noticed there was a bug in the skyshine extraction that lead to an increase in the observed skyshine events.

### 2. Updated sideband analysis for publication
These cells adapt the above machinery with the CCinc tune / re-weighting so that the same MicroBooNE tune is applied evenly across all analyses.

Both adopt the CCinc dirt scaling factor of $\alpha_{\text{dirt}} = 0.18$. We can then vary the dirt rate around to see if it agrees with the CCinc scaling, which provides an independent assessment for the dirt interaction rate and helps to define the dirt systematic uncertainty.

---------------
## 1. Thesis analysis

Important to note that these cells provide the sideband analysis AFTER the introduction of the skyshine neutrons, the CCinc dirt scaling, and the restriction of the FV from r = 1m, y = 1.5m to r = 0.7m, y = 1m.

### Load MC samples

In [ ]:
MC_directory = '../GENIE_CV_tilt_MC_ToolChain/'
file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

# POT for World samples --> 5.518e16 POT per file
POT = 2.283e20     # MC POT (modify if needed, calculated based on total number of files present)

# scalar accumulators
MC_cluster_number    = []
MC_cluster_time      = []
MC_cluster_charge    = []
MC_cluster_Qb        = []
MC_cluster_Hits      = []
MC_TankMRDCoinc      = []
MC_NoVeto            = []
MC_recoVtx           = []
MC_recoVty           = []
MC_recoVtz           = []
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_bunchTimes        = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_QEL               = []
MC_MEC               = []
MC_PDG               = []
MC_Z                 = []
MC_file_index        = []   # which file each cluster came from
MC_event_number      = []

# jagged accumulators
MC_hitX  = []
MC_hitY  = []
MC_hitZ  = []
MC_hitT  = []
MC_hitPE = []
MC_hitID = []
MC_MRD_hitT  = []
MC_FMV_hitT  = []

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    MC_cluster_number.append(T['clusterNumber'].array(library='np'))
    MC_cluster_charge.append(T['clusterPE'].array(library='np'))
    MC_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    MC_cluster_Hits.append(T['clusterHits'].array(library='np'))
    MC_cluster_time.append(T['clusterTime'].array(library='np'))
    MC_bunchTimes.append(T['bunchTimes'].array(library='np'))
    MC_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    MC_NoVeto.append(T['NoVeto'].array(library='np'))
    MC_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    MC_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    MC_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))
    MC_event_number.append(CEN)
    MC_file_index.append(np.full(len(CEN), file_idx, dtype=np.int32))

    MC_hitX.append(T['hitX'].array())
    MC_hitY.append(T['hitY'].array())
    MC_hitZ.append(T['hitZ'].array())
    MC_hitT.append(T['hitT'].array())
    MC_hitPE.append(T['hitPE'].array())
    MC_hitID.append(T['hitDetID'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)

    # map trigger-level quantities onto clusters via CEN
    MC_Vtx.append(vx_cm[CEN])
    MC_Vty.append(vy_cm[CEN])
    MC_Vtz.append(vz_cm[CEN])
    MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np')[CEN])
    MC_External.append(ext_flags[CEN])
    MC_FV.append(fv_flags[CEN])
    MC_NC.append(S['trueNC'].array(library='np')[CEN])
    MC_QEL.append(S['trueQEL'].array(library='np')[CEN])
    MC_MEC.append(S['trueMEC'].array(library='np')[CEN])
    MC_PDG.append(S['trueFSLPdg'].array(library='np')[CEN])
    MC_Z.append(S['trueTargetZ'].array(library='np')[CEN])

    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    MC_MRD_hitT.append(mrd_t[CEN])
    MC_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_cluster_number  = np.concatenate(MC_cluster_number)
MC_cluster_time    = np.concatenate(MC_cluster_time)
MC_cluster_charge  = np.concatenate(MC_cluster_charge)
MC_cluster_Qb      = np.concatenate(MC_cluster_Qb)
MC_cluster_Hits    = np.concatenate(MC_cluster_Hits)
MC_bunchTimes      = np.concatenate(MC_bunchTimes)
MC_TankMRDCoinc    = np.concatenate(MC_TankMRDCoinc)
MC_NoVeto          = np.concatenate(MC_NoVeto)
MC_recoVtx         = np.concatenate(MC_recoVtx)
MC_recoVty         = np.concatenate(MC_recoVty)
MC_recoVtz         = np.concatenate(MC_recoVtz)
MC_event_number    = np.concatenate(MC_event_number)
MC_file_index      = np.concatenate(MC_file_index)
MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_FV              = np.concatenate(MC_FV)
MC_NC              = np.concatenate(MC_NC)
MC_QEL             = np.concatenate(MC_QEL)
MC_MEC             = np.concatenate(MC_MEC)
MC_PDG             = np.concatenate(MC_PDG)
MC_Z               = np.concatenate(MC_Z)

MC_hitX     = ak.concatenate(MC_hitX)
MC_hitY     = ak.concatenate(MC_hitY)
MC_hitZ     = ak.concatenate(MC_hitZ)
MC_hitT     = ak.concatenate(MC_hitT)
MC_hitPE    = ak.concatenate(MC_hitPE)
MC_hitID    = ak.concatenate(MC_hitID)
MC_MRD_hitT = ak.concatenate(MC_MRD_hitT)
MC_FMV_hitT = ak.concatenate(MC_FMV_hitT)

print(f'POT: {POT:.3e}')
print(f'Total clusters: {len(MC_cluster_number)}')
print('\ndone')

### Load On-Beam Data

In [ ]:
def extract_run_number(file_name):
    return int(file_name.split('_')[0][1:])  # Split by '_' and remove 'R' to get the run number

directory = '../FY22_23/'
file_names = [file for file in os.listdir(directory) if os.path.isfile(os.path.join(directory, file))]
print('There are: ', len(file_names), ' files\n')

# POT file (contains total POT and triggers per run)
csv_file = '../POT_files/POT_trigger_FY22_23_summary.csv'  # make sure you have a POT summary file
pot_df = pd.read_csv(csv_file)

# extract total POT and triggers
total_POT_TOR860 = sum(pot_df["TOR860_POTe12"])/(1e8)   # convert to e20 POT
total_triggers = sum(pot_df["TOTAL_TRIGGERS"])

cluster_time = []
cluster_charge = []
cluster_Hits = []
cluster_Qb = []
TankMRDCoinc = []
NoVeto = []
only_prompt_cluster = []
MRD_activity = []
Veto_activity = []
Vtx_reco = []
Vty_reco = []
Vtz_reco = []
hitX = []
hitY = []
hitZ = []
hitT = []
hitPE = []
runs = []

counter = 0
print('(', (counter), '/', len(file_names), ')')

for file_name in file_names:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names), ')')
    
    with uproot.open(os.path.join(directory, file_name)) as file:
        
        Event = file["data"]
        
        runs.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time.append(Event["cluster_time_BRF"].array(library="np"))
        cluster_charge.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity.append(Event["MRD_activity"].array(library="np"))
        Veto_activity.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco.append(Event["recoVtx"].array(library="np"))
        Vty_reco.append(Event["recoVty"].array(library="np"))
        Vtz_reco.append(Event["recoVtz"].array(library="np"))
        
        hitX.append(Event["hitX"].array())
        hitY.append(Event["hitY"].array())
        hitZ.append(Event["hitZ"].array())
        hitT.append(Event["hitT"].array())
        hitPE.append(Event["hitPE"].array())
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs = np.concatenate(runs)
cluster_time = np.concatenate(cluster_time)
cluster_charge = np.concatenate(cluster_charge)
cluster_Qb = np.concatenate(cluster_Qb)
cluster_Hits = np.concatenate(cluster_Hits)
TankMRDCoinc = np.concatenate(TankMRDCoinc)
NoVeto = np.concatenate(NoVeto)
only_prompt_cluster = np.concatenate(only_prompt_cluster)
MRD_activity = np.concatenate(MRD_activity)
Veto_activity = np.concatenate(Veto_activity)
Vtx_reco = np.concatenate(Vtx_reco)
Vty_reco = np.concatenate(Vty_reco)
Vtz_reco = np.concatenate(Vtz_reco)

# Jagged arrays
hitX = ak.concatenate(hitX)
hitY = ak.concatenate(hitY)
hitZ = ak.concatenate(hitZ)
hitT = ak.concatenate(hitT)
hitPE = ak.concatenate(hitPE)

print(total_triggers, 'triggers')
print(total_POT_TOR860, 'e20 POT\n')

### Load Off-Beam Data

In [ ]:
directory_ext = '../off_beam/'
file_names_ext = [file for file in os.listdir(directory_ext) if os.path.isfile(os.path.join(directory_ext, file))]
print('There are: ', len(file_names_ext), ' files\n')

# POT equivalent file for off-beam (just total number of triggers)
csv_file_ext = '../POT_files/offbeam_summary.csv'
pot_df_ext = pd.read_csv(csv_file_ext)

# extract total POT and triggers
total_ext_triggers = sum(pot_df_ext["TOTAL_TRIGGERS"])

cluster_time_ext = []
cluster_charge_ext = []
cluster_Hits_ext = []
cluster_Qb_ext = []
TankMRDCoinc_ext = []
NoVeto_ext = []
only_prompt_cluster_ext = []
MRD_activity_ext = []
Veto_activity_ext = []
Vtx_reco_ext = []
Vty_reco_ext = []
Vtz_reco_ext = []
hitX_ext = []
hitY_ext = []
hitZ_ext = []
hitT_ext = []
hitPE_ext = []
runs_ext = []

counter = 0
print('(', (counter), '/', len(file_names_ext), ')')

for file_name in file_names_ext:
    
    run_number = extract_run_number(file_name)
    counter += 1
    
    if counter % 50 == 0:
        print('(', (counter), '/', len(file_names), ')')
    
    with uproot.open(os.path.join(directory_ext, file_name)) as file:
        
        Event = file["data"]
        
        runs_ext.append(Event["run_number"].array(library="np"))
        
        # needed for both
        cluster_time_ext.append(Event["cluster_time"].array(library="np"))
        cluster_charge_ext.append(Event["cluster_PE"].array(library="np"))
        cluster_Qb_ext.append(Event["cluster_Qb"].array(library="np"))
        cluster_Hits_ext.append(Event["cluster_Hits"].array(library="np"))
        TankMRDCoinc_ext.append(Event["TankMRDCoinc"].array(library="np"))
        NoVeto_ext.append(Event["NoVeto"].array(library="np"))
        only_prompt_cluster_ext.append(Event["only_prompt_cluster"].array(library="np"))
        MRD_activity_ext.append(Event["MRD_activity"].array(library="np"))
        Veto_activity_ext.append(Event["FMV_activity"].array(library="np"))
        Vtx_reco_ext.append(Event["recoVtx"].array(library="np"))
        Vty_reco_ext.append(Event["recoVty"].array(library="np"))
        Vtz_reco_ext.append(Event["recoVtz"].array(library="np"))
        
        hitX_ext.append(Event["hitX"].array())
        hitY_ext.append(Event["hitY"].array())
        hitZ_ext.append(Event["hitZ"].array())
        hitT_ext.append(Event["hitT"].array())
        hitPE_ext.append(Event["hitPE"].array())
        
        
# Concatenate everything once at the end
print('\nManaging arrays...\n')
runs_ext = np.concatenate(runs_ext)
cluster_time_ext = np.concatenate(cluster_time_ext)
cluster_charge_ext = np.concatenate(cluster_charge_ext)
cluster_Qb_ext = np.concatenate(cluster_Qb_ext)
cluster_Hits_ext = np.concatenate(cluster_Hits_ext)
TankMRDCoinc_ext = np.concatenate(TankMRDCoinc_ext)
NoVeto_ext = np.concatenate(NoVeto_ext)
only_prompt_cluster_ext = np.concatenate(only_prompt_cluster_ext)
MRD_activity_ext = np.concatenate(MRD_activity_ext)
Veto_activity_ext = np.concatenate(Veto_activity_ext)
Vtx_reco_ext = np.concatenate(Vtx_reco_ext)
Vty_reco_ext = np.concatenate(Vty_reco_ext)
Vtz_reco_ext = np.concatenate(Vtz_reco_ext)

# Jagged arrays
hitX_ext = ak.concatenate(hitX_ext)
hitY_ext = ak.concatenate(hitY_ext)
hitZ_ext = ak.concatenate(hitZ_ext)
hitT_ext = ak.concatenate(hitT_ext)
hitPE_ext = ak.concatenate(hitPE_ext)

print(total_ext_triggers, 'triggers')

### Load Skyshine simulation samples

Skyshine based on modeling as outlined in Steven Doran's thesis. Prior to plotting sideband plots we can constrain the skyshine rate by looking at the rise in activity later into the spill as compared to the MC prediction.

In [ ]:
file_name = '../skyshine_100k_no_ext_clusters.root'

# scalar accumulators
skyshine_cluster_number    = []
skyshine_cluster_time      = []
skyshine_cluster_charge    = []
skyshine_cluster_Qb        = []
skyshine_cluster_Hits      = []
skyshine_TankMRDCoinc      = []
skyshine_NoVeto            = []
skyshine_recoVtx           = []
skyshine_recoVty           = []
skyshine_recoVtz           = []
skyshine_event_number      = []

# jagged accumulators
skyshine_hitX  = []
skyshine_hitY  = []
skyshine_hitZ  = []
skyshine_hitT  = []
skyshine_hitPE = []
skyshine_hitID = []
skyshine_MRD_hitT  = []
skyshine_FMV_hitT  = []

path = os.path.join(file_name)
root = uproot.open(path)

# ── cluster tree ──────────────────────────────────────────────────────────
T   = root['phaseIITankClusterTree']
CEN = T['eventNumber'].array(library='np')

skyshine_cluster_number.append(T['clusterNumber'].array(library='np'))
skyshine_cluster_charge.append(T['clusterPE'].array(library='np'))
skyshine_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
skyshine_cluster_Hits.append(T['clusterHits'].array(library='np'))
skyshine_cluster_time.append(T['clusterTime'].array(library='np'))
skyshine_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
skyshine_NoVeto.append(T['NoVeto'].array(library='np'))
skyshine_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
skyshine_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
skyshine_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))
skyshine_event_number.append(CEN)

skyshine_hitX.append(T['hitX'].array())
skyshine_hitY.append(T['hitY'].array())
skyshine_hitZ.append(T['hitZ'].array())
skyshine_hitT.append(T['hitT'].array())
skyshine_hitPE.append(T['hitPE'].array())
skyshine_hitID.append(T['hitDetID'].array())

# ── trigger tree ──────────────────────────────────────────────────────────
S   = root['phaseIITriggerTree']

# map trigger-level quantities onto clusters via CEN
mrd_t   = S['MRDhitT'].array()
veto_t  = S['FMVhitT'].array()
skyshine_MRD_hitT.append(mrd_t[CEN])
skyshine_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

skyshine_cluster_number  = np.concatenate(skyshine_cluster_number)
skyshine_cluster_time    = np.concatenate(skyshine_cluster_time)
skyshine_cluster_charge  = np.concatenate(skyshine_cluster_charge)
skyshine_cluster_Qb      = np.concatenate(skyshine_cluster_Qb)
skyshine_cluster_Hits    = np.concatenate(skyshine_cluster_Hits)
skyshine_TankMRDCoinc    = np.concatenate(skyshine_TankMRDCoinc)
skyshine_NoVeto          = np.concatenate(skyshine_NoVeto)
skyshine_recoVtx         = np.concatenate(skyshine_recoVtx)
skyshine_recoVty         = np.concatenate(skyshine_recoVty)
skyshine_recoVtz         = np.concatenate(skyshine_recoVtz)
skyshine_event_number    = np.concatenate(skyshine_event_number)

skyshine_hitX     = ak.concatenate(skyshine_hitX)
skyshine_hitY     = ak.concatenate(skyshine_hitY)
skyshine_hitZ     = ak.concatenate(skyshine_hitZ)
skyshine_hitT     = ak.concatenate(skyshine_hitT)
skyshine_hitPE    = ak.concatenate(skyshine_hitPE)
skyshine_hitID    = ak.concatenate(skyshine_hitID)
skyshine_MRD_hitT = ak.concatenate(skyshine_MRD_hitT)
skyshine_FMV_hitT = ak.concatenate(skyshine_FMV_hitT)

print(f'Total clusters: {len(skyshine_cluster_number)}')
print('\ndone')

### Apply Selection cuts

In [ ]:
# First, MC

# the dimensions of the saved arrays are as follows:
# [0]: nuNCQE (signal)
# [1]: NCother
# [2]: CC
# [3]: nubarNCQE
# [4]: Out-of-FV (OOFV)
# [5]: External (interactions outside of the water tank entirely)

MC_reco_E = [[], [], [], [], [], []]
MC_CV_x = [[], [], [], [], [], []]
MC_CV_y = [[], [], [], [], [], []]
MC_CV_z = [[], [], [], [], [], []]
MC_bunch_times = [[], [], [], [], [], []]
MC_vertex_x = [[], [], [], [], [], []]
MC_vertex_y = [[], [], [], [], [], []]
MC_vertex_z = [[], [], [], [], [], []]
MC_hits = [[], [], [], [], [], []]


for i in trange(len(MC_cluster_time), desc = 'applying NCQE selection cuts to MC...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if MC_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if MC_bunchTimes[i] > bunch_time_cutoff + time_to_prompt_end:
        continue
    # minimum 10 hits
    if MC_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    # No MRD/FMV correlated activity
    if (
        MC_TankMRDCoinc[i] == 1
        or MC_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in MC_MRD_hitT[i])
        or any(0 <= t <= 300 for t in MC_FMV_hitT[i])
    ):
        continue
        
    # Cluster time within 1.6 us beam spill
    if MC_bunchTimes[i] > bunch_time_cutoff:
        continue
        
    # Add earlier time cut (reject anything later than the 25th bunch center)
    # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
    #if MC_bunchTimes[i] > centers[25]:
    #    continue
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(MC_cluster_time)):        # look at the subsequent clusters in this event
        if MC_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if MC_bunchTimes[k] < bunch_time_cutoff + time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if MC_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut (FV defined in first cell)
    if MC_recoVtx[i] < -50:
        continue
    r2 = MC_recoVtx[i]**2 + MC_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(MC_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(MC_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    # INVERT BUNCH CUT FOR SIDEBAND
    if bunch_cutting(MC_bunchTimes[i], centers) == True:
        continue
        
    cw = pairwise_relative_direction(MC_hitX[i], MC_hitY[i], MC_hitZ[i], MC_hitPE[i], MC_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
        
    # event categorization
    if MC_External[i] == True:
        indy = 5
        MC_reco_E[indy].append(recoE)
        MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
        MC_bunch_times[indy].append(MC_bunchTimes[i])
        MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
        MC_hits[indy].append(MC_cluster_Hits[i])
    else:
        if MC_FV[i] == True:
            if MC_NC[i] == 1:       # NC
                if MC_QEL[i] == 1:
                    if MC_Z[i] == 8:
                        if MC_PDG[i] == 14 or MC_PDG[i] == 12:      # nu (mu + e)
                            # nuNCQE
                            indy = 0
                            MC_reco_E[indy].append(recoE)
                            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                            MC_bunch_times[indy].append(MC_bunchTimes[i])
                            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                            MC_hits[indy].append(MC_cluster_Hits[i])
                        elif MC_PDG[i] == -14 or MC_PDG[i] == -12:
                            # nubarNCQE
                            indy = 3
                            MC_reco_E[indy].append(recoE)
                            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                            MC_bunch_times[indy].append(MC_bunchTimes[i])
                            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                            MC_hits[indy].append(MC_cluster_Hits[i])
                    else:
                        # NC elastic on H (NCother)
                        indy = 1
                        MC_reco_E[indy].append(recoE)
                        MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                        MC_bunch_times[indy].append(MC_bunchTimes[i])
                        MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                        MC_hits[indy].append(MC_cluster_Hits[i])
                else:
                    # NC, not QEL (NCother)
                    indy = 1
                    MC_reco_E[indy].append(recoE)
                    MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                    MC_bunch_times[indy].append(MC_bunchTimes[i])
                    MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                    MC_hits[indy].append(MC_cluster_Hits[i])
            else:
                # CC
                indy = 2
                MC_reco_E[indy].append(recoE)
                MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                MC_bunch_times[indy].append(MC_bunchTimes[i])
                MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                MC_hits[indy].append(MC_cluster_Hits[i])
        else:
            # out-of-FV
            indy = 4
            MC_reco_E[indy].append(recoE)
            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
            MC_bunch_times[indy].append(MC_bunchTimes[i])
            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
            MC_hits[indy].append(MC_cluster_Hits[i])
          

print('done')

In [ ]:
# Apply selection cuts for On-Beam data

reco_E = []
CV_x = []
CV_y = []
CV_z = []
bunch_times = []
vertex_x = []
vertex_y = []
vertex_z = []
hits = []
reco_runs = []


for i in trange(len(cluster_time), desc = 'applying NCQE selection cuts to on-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc[i] == 1 or NoVeto[i] == 0 or MRD_activity[i] != 0 or Veto_activity[i] != 0:
        continue
        
    if runs[i] > 4000:   # FY23
        if cluster_time[i] < spill_start_23:
            continue
        if cluster_time[i] > spill_end_23:
            continue
            
        # add for earlier time
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
        #if cluster_time[i] > centers_data_23[25]:
        #    continue
        
        # INVERT BUNCH CUT FOR SIDEBAND
        if bunch_cutting(cluster_time[i], centers_data_23) == True:
            continue
    else:
        if cluster_time[i] < spill_start_22:
            continue
        if cluster_time[i] > spill_end_22:
            continue
            
        # add for earlier time
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
        #if cluster_time[i] > centers_data_22[25]:
        #    continue
        
        # INVERT BUNCH CUT FOR SIDEBAND
        if bunch_cutting(cluster_time[i], centers_data_22) == True:
            continue
        
    # CCB cut
    if cluster_Qb[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco[i] < -50:
        continue
    r2 = Vtx_reco[i]**2 + Vtz_reco[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX[i], hitY[i], hitZ[i], hitPE[i], hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    # event selection
    reco_E.append(recoE)
    CV_x.append(cw[0]); CV_y.append(cw[1]); CV_z.append(cw[2])
    bunch_times.append(cluster_time[i])
    vertex_x.append(Vtx_reco[i]); vertex_y.append(Vty_reco[i]); vertex_z.append(Vtz_reco[i])
    hits.append(cluster_Hits[i])
    reco_runs.append(runs[i])
          

print('done')

In [ ]:
# Apply it for off-beam data

ext_reco_E = []
ext_CV_x = []
ext_CV_y = []
ext_CV_z = []
ext_bunch_times = []
ext_vertex_x = []
ext_vertex_y = []
ext_vertex_z = []
ext_hits = []
ext_reco_runs = []

for i in trange(len(cluster_time_ext), desc = 'applying NCQE selection cuts to off-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster_ext[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc_ext[i] == 1 or NoVeto_ext[i] == 0 or MRD_activity_ext[i] != 0 or Veto_activity_ext[i] != 0:
        continue
        
    if runs_ext[i] > 4000:   # FY23
        if cluster_time_ext[i] < spill_start_23:
            continue
        if cluster_time_ext[i] > spill_end_23:
            continue
            
        # add for earlier time
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
        #if cluster_time_ext[i] > centers_data_23[25]:
        #    continue
        
        # INVERT BUNCH CUT FOR SIDEBAND
        if bunch_cutting(cluster_time_ext[i], centers_data_23) == True:
            continue
    else:
        if cluster_time_ext[i] < spill_start_22:
            continue
        if cluster_time_ext[i] > spill_end_22:
            continue
            
        # add for earlier time
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
        #if cluster_time_ext[i] > centers_data_22[25]:
        #    continue
        
        # INVERT BUNCH CUT FOR SIDEBAND
        if bunch_cutting(cluster_time_ext[i], centers_data_22) == True:
            continue
        
    # CCB cut
    if cluster_Qb_ext[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco_ext[i] < -50:
        continue
    r2 = Vtx_reco_ext[i]**2 + Vtz_reco_ext[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco_ext[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge_ext[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX_ext[i], hitY_ext[i], hitZ_ext[i], hitPE_ext[i], hitT_ext[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
        
    # event selection
    ext_reco_E.append(recoE)
    ext_CV_x.append(cw[0]); ext_CV_y.append(cw[1]); ext_CV_z.append(cw[2])
    ext_bunch_times.append(cluster_time_ext[i])
    ext_vertex_x.append(Vtx_reco_ext[i]); ext_vertex_y.append(Vty_reco_ext[i]); ext_vertex_z.append(Vtz_reco_ext[i])
    ext_hits.append(cluster_Hits_ext[i])
    ext_reco_runs.append(runs_ext[i])
          

print('done')

In [ ]:
# And lastly, apply it to our skyshine MC sample
# (selection cuts are revised as its not a full GENIE simulation with timing)

skyshine_reco_E = []
skyshine_CV_x = []
skyshine_CV_y = []
skyshine_CV_z = []
skyshine_bunch_times = []
skyshine_vertex_x = []
skyshine_vertex_y = []
skyshine_vertex_z = []
skyshine_hits = []


# do not apply timing cuts
for i in trange(len(skyshine_cluster_time), desc = 'applying NCQE selection cuts to simulated skyshine...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if skyshine_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if skyshine_cluster_time[i] > bunch_time_cutoff + time_to_prompt_end:
        continue
    # minimum 10 hits
    if skyshine_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    if (
        skyshine_TankMRDCoinc[i] == 1
        or skyshine_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in skyshine_MRD_hitT[i])
        or any(0 <= t <= 300 for t in skyshine_FMV_hitT[i])
    ):
        continue
        
    # Cluster time within 1.6 us beam spill of particle creation (t0 = 0 for these simulations)
    if skyshine_cluster_time[i] > bunch_time_cutoff:
        continue
        
    # No bunch cutting applied
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(skyshine_cluster_time)):        # look at the subsequent clusters in this event
        if skyshine_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if skyshine_cluster_time[k] < bunch_time_cutoff + time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if skyshine_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut
    if skyshine_recoVtx[i] < -50:
        continue
    r2 = skyshine_recoVtx[i]**2 + skyshine_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(skyshine_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(skyshine_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(skyshine_hitX[i], skyshine_hitY[i], skyshine_hitZ[i], skyshine_hitPE[i], skyshine_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    skyshine_reco_E.append(recoE)
    skyshine_CV_x.append(cw[0]); skyshine_CV_y.append(cw[1]); skyshine_CV_z.append(cw[2])
    skyshine_bunch_times.append(skyshine_cluster_time[i])
    skyshine_vertex_x.append(skyshine_recoVtx[i]); 
    skyshine_vertex_y.append(skyshine_recoVty[i]);
    skyshine_vertex_z.append(skyshine_recoVtz[i])
    skyshine_hits.append(skyshine_cluster_Hits[i])
    

print('done')

### Load functions and helpful information (plotting, analysis)

In [ ]:
# 1. Scaling factors
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
POT_MC = POT/(1e20)
POT_DATA = total_POT_TOR860
TRIG_DATA = total_triggers
TRIG_EXT = total_ext_triggers


# 2. Chain all bunch times to a common t0
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
def get_normalized_times(times, runs, mc=False):
    norm_times = []
    for t, r in zip(times, runs):
        if mc:
            t0 = centers[0]
        else:
            t0 = centers_data_22[0] if r < 4000 else centers_data_23[0]
        
        tnorm = t - t0
        # Filter to only include the 80 bunches (approx 18.936ns * 80)
        if 0 <= tnorm < (80 * 18.936):
            norm_times.append(tnorm)
    return np.array(norm_times)

# 1. Process MC (Categories 0-5)
bunches_80_mc = [get_normalized_times(MC_bunch_times[i], [0]*len(MC_bunch_times[i]), mc=True) 
                 for i in range(6)]

# 2. Process On-Beam Data
bunches_80_data = get_normalized_times(bunch_times, reco_runs)

# 3. Process Off-Beam (External)
bunches_80_ext = get_normalized_times(ext_bunch_times, ext_reco_runs)


# 3. Calculation of radial distance of the reco vertex
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
# For MC (6 categories)
radius_mc = [np.sqrt(np.array(MC_vertex_x[i])**2 + np.array(MC_vertex_z[i])**2) for i in range(6)]
# For On-Beam Data
radius_data = np.sqrt(np.array(vertex_x)**2 + np.array(vertex_z)**2)
# For Off-Beam (EXT) Data
radius_ext = np.sqrt(np.array(ext_vertex_x)**2 + np.array(ext_vertex_z)**2)


# 4. Relative time from nearest bunch center
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
def distance_to_bunch_center(cluster_time, centers):
    avg_diff = 0.0588
    # Ensure 'centers' is defined in your environment
    shifted_centers = centers - avg_diff 
    return np.min(np.abs(shifted_centers - cluster_time))

# --- MC Calculation (Static) ---
# Assuming 'centers' is your MC bunch center array
delta_t_mc = [
    np.array([distance_to_bunch_center(t, centers) for t in cat]) 
    for cat in MC_bunch_times
]

# --- On-Beam Data (Run Dependent) ---
delta_t_data_list = []
for t, run in zip(bunch_times, reco_runs):
    # Select epoch centers based on run number
    current_centers = centers_data_22 if run < 4000 else centers_data_23
    delta_t_data_list.append(distance_to_bunch_center(t, current_centers))

delta_t_data = np.array(delta_t_data_list)

# --- Off-Beam Data (Run Dependent) ---
delta_t_ext_list = []
for t, run in zip(ext_bunch_times, ext_reco_runs):
    # Same logic for Off-beam/External data
    current_centers = centers_data_22 if run < 4000 else centers_data_23
    delta_t_ext_list.append(distance_to_bunch_center(t, current_centers))

delta_t_ext = np.array(delta_t_ext_list)


# 5. Calculate skyshine rate (t_cut corresponds to the 1/3 spill cut)
'''
# Skyshine fit results From my original analysis (5-bunch bin):
calculate_skyshine_in_window(
    t_cut=centers_data_23[25]-centers[0], 
    t0=340.8, err_t0=58.6, 
    A=0.000981, err_A=0.000128
)
Time Cut: 638.6738551505571 ns
Expected Skyshine Events: 68.3 ± 28.2
'''
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * 
# BUG: t_cut = centers_data_23[25]-centers[0] in the old scripts
# it has been corrected here --> the impact was an overestimation of 
# the skyshine rate. centers[] is the MC array
def calculate_delayed_skyshine_rate(bins, prediction_data, weights, on_beam_data, 
                                    t_cut=centers_data_23[25]-centers_data_23[0], DEBUG=False):
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_width = bins[1] - bins[0]  
    
    # 1. Build Histograms (Same as before)
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    data_err[data_err == 0] = 1.0 
    
    pred_counts = np.zeros_like(data_counts, dtype=float)
    pred_err_sq = np.zeros_like(data_counts, dtype=float)
    for i in range(len(prediction_data)):
        c, _ = np.histogram(prediction_data[i], bins=bins, weights=weights[i])
        w2, _ = np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)
        pred_counts += c
        pred_err_sq += w2
    pred_err = np.sqrt(pred_err_sq)
    pred_err[pred_err == 0] = 1.0

    # 2. Define the Piecewise "Delayed Rise" Model
    def delayed_linear(t, C, m, t0):
        # C: Flat baseline events/bin
        # m: Skyshine slope
        # t0: Turn-on time in ns
        return np.where(t < t0, C, C + m * (t - t0))

    # 3. Perform the Fits
    # p0 guesses: Baseline=100, slope=0.2, turn-on=400 ns
    popt_data, pcov_data = curve_fit(
        delayed_linear, bin_centers, data_counts, 
        sigma=data_err, absolute_sigma=True, 
        p0=[100.0, 0.2, 400.0],
        bounds=([0, 0, 0], [np.inf, np.inf, 1500]) # Keep parameters physical
    )
    
    # For prediction, we expect slope ~ 0, so bounds allow standard flat line
    popt_pred, pcov_pred = curve_fit(
        delayed_linear, bin_centers, pred_counts, 
        sigma=pred_err, absolute_sigma=True,
        p0=[100.0, 0.0, 400.0],
        bounds=([0, -np.inf, 0], [np.inf, np.inf, 1500])
    )

    C_data, m_data, t0_data = popt_data
    C_pred, m_pred, t0_pred = popt_pred
    
    err_m_data = np.sqrt(pcov_data[1, 1])
    err_m_pred = np.sqrt(pcov_pred[1, 1])
    err_t0_data = np.sqrt(pcov_data[2, 2])

    # 4. Calculate True Rate
    delta_m = m_data - m_pred
    err_delta_m = np.sqrt(err_m_data**2 + err_m_pred**2)
    rate_increase = delta_m / bin_width
    err_rate_increase = err_delta_m / bin_width
    
    
    # - - - - - - - - - - - - - - - - -
    # Calculate the (extrapolated)/expected rate within the signal window
    # As the sideband has a different exposure, we need to calculate an appropriate
    # scale factor between the sideband and signal
    
    # +/-1 sigma cut = 3.59ns
    # Full spill be 81 bunches * 3.59*2 = 581.58ns total —> for the first 25 bunches (1/3 sill cut) it is: 179.5ns
    
    # The sideband was the inter-bunch regions for a total of 80 intervals
    # The bunches are spaced by 18.936 ns
    # The sideband therefore only explores 18.936/2 - 3.59*2 = 2.288ns per “half” 
    # —> multiply by 2 because we’re exploring the space between the bunches = 4.576 x 25 bunches = 114.4ns
    # + a “little” of the first bunch since we cut off the start of the spill ~9ns before the first bunch
    # —> call it 115ns for the sideband
    
    # Therefore the scaling factor is ~179.5ns / 115ns = 1.56 (assuming first 1/3 spill cut)
    EXPOSURE_SCALE = 1.56
    
    delta_t = t_cut - t0_data
    
    # Calculate expected events
    N_events = 0.5 * rate_increase * (delta_t**2)
    
    # Error propagation
    # Partial derivative wrt A: 0.5 * delta_t^2
    term_A = (0.5 * (delta_t**2) * err_rate_increase)**2
    # Partial derivative wrt t0: -A * delta_t
    term_t0 = (rate_increase * delta_t * err_t0_data)**2
    
    err_N = np.sqrt(term_A + term_t0)
    
    if DEBUG:
        print("\n" + "="*50)
        print(" DELAYED SKYSHINE EXTRACTION (PIECEWISE)")
        print("="*50)
        print(f"Data Skyshine Turn-on Time: {t0_data:.1f} ± {err_t0_data:.1f} ns")
        print(f"Data Baseline (C):          {C_data:.1f} events/bin")
        print(f"Data Slope after t0:        {m_data:.5f} ± {err_m_data:.5f} (evts/bin)/ns")
        print("-" * 50)
        print(f"Excess Slope:               {delta_m:.5f} ± {err_delta_m:.5f} (evts/bin)/ns")
        print(f"True Skyshine Acceleration: {rate_increase:.6f} ± {err_rate_increase:.6f} events/ns²")
        print(f"Time Cut: {t_cut} ns")
        print(f"Expected Skyshine Events in sideband region: {N_events:.1f} ± {err_N:.1f}")
        print(f"Fit Error: {100*err_N/N_events:.1f}%")
        print("="*50 + "\n")
        
        print("*"*50)
        print(" EXTRAPOLATED TO SIGNAL BAND")
        print("*"*50)
        print(f"Exposure Scaling: {EXPOSURE_SCALE}")
        print(f"Time Cut: {t_cut} ns")
        print(f"Expected Skyshine Events: {EXPOSURE_SCALE*N_events:.1f} ± {EXPOSURE_SCALE*err_N:.1f}")
        print(f"Fit Error: {100*err_N/N_events:.1f}%")
        print("*"*50 + "\n")
        
    
    return popt_data


# 7. Plotting function
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
def plot_sideband_comparison(bins, mc_data, on_beam_data, off_beam_data, 
                             pot_mc, pot_data, trig_ext, trig_data,
                             axis_label, y_axis_label, x_limit, y_limit, xaxis_minor, 
                             plot_name, legend_columns=3, 
                             DEBUG=False, ALPHA_SCALE=0.18, CALCULATE_SKYSHINE=False, PLOT_SKYSHINE_FIT=False,
                             SAVE_PLOT=False):
    
    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    
    # 2. Prepare Prediction Stack (MC + Off-Beam)
    # Order: [Ext_MC, OOF, CC, NCother, nubarNCQE, nuNCQE, Off-Beam Data]
    prediction_data = mc_data + [off_beam_data]
    
    weights = [np.ones(len(arr)) * f_pot for arr in mc_data]
    weights.append(np.ones(len(off_beam_data)) * f_ext)
    
    # --- DEBUG PRINTS ---
    if DEBUG:
        print(f"f_pot scaling factor: {f_pot:.3f}")
        print(f"Raw Dirt events before weights: {len(prediction_data[0])}")
        print(f"Dirt Yield BEFORE scale: {np.sum(weights[0]):.1f}")
    
    # 3. Aesthetics
    labels_base = ['External (MC)', 'Out-of-FV', 'CC', 'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors = ['papayawhip', 'lime', 'darkred', 'cyan', 'darkorange', '#0066ff', 'gray']
    # Match outline colors to the fill (or use 'navy' as in your original script)
    color_outline = ['navy'] * 6 + ['black']
    
    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * ALPHA_SCALE
            if DEBUG:
                print(f"Applied {ALPHA_SCALE} scale to index {i} ({label})")
    
    if DEBUG:
        print(f"Dirt Yield AFTER scale: {np.sum(weights[0]):.1f}")
        print(f"Off-Beam Yield: {np.sum(weights[-1]):.1f}")
    
    
    # skyshine rate calculation
    # ----------------------------------------------
    if CALCULATE_SKYSHINE:
        popt_data = calculate_delayed_skyshine_rate(bins, prediction_data, weights, on_beam_data, DEBUG=True)
        C_data, m_data, t0_data = popt_data  # Unpack the best-fit parameters

        # --- NEW: Calculate Chi-Squared for the Fit ---
        # 1. Evaluate the fit function at the center of every bin
        bin_centers = 0.5 * (bins[1:] + bins[:-1])
        fit_at_centers = np.where(bin_centers < t0_data, C_data, C_data + m_data * (bin_centers - t0_data))

        # 2. Get data counts and errors to compare against
        data_counts, _ = np.histogram(on_beam_data, bins=bins)
        data_err = np.sqrt(data_counts)

        # 3. Mask out empty bins to avoid division by zero in the chi2 math
        valid_bins = data_counts > 0 

        # 4. Calculate chi2 and Degrees of Freedom (Ndof)
        chi2 = np.sum( ((data_counts[valid_bins] - fit_at_centers[valid_bins]) / data_err[valid_bins])**2 )
        ndof = np.sum(valid_bins) - 3  # 3 fit parameters: C, m, t0
        if DEBUG:
            print('Chi^2 =', chi2, ' ndof =', ndof, ' reduced Chi^2 =', chi2/ndof)
        
    # ----------------------------------------------
    
    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]
    
    # 4. Math for Residuals and Errors
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths = np.diff(bins)
    
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    
    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))

    # --- Plotting ---
    fig = plt.figure(figsize=(7, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill (No edges to avoid the bar-graph look)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            label=labels, color=colors, linewidth=0)

    # B. The Outline (Reverting to your original 'step' logic for clean category boundaries)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            histtype='step', color=color_outline, linewidth=0.8)

    # C. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2, 
                fmt='ko', label='On-Beam Data', markersize=4, capsize=0)

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    
    # Error propagation for (D/P - 1)
    # sigma_res = (D/P) * sqrt( (sigD/D)^2 + (sigP/P)^2 )
    ratio = data_counts[mask] / pred_counts[mask]
    # Handle division by zero for error calculation if data_counts is 0
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
    
    ### NEW: Overlay the Piecewise Fit
    # Create a dense array of x points spanning the plot window
    x_fit = np.linspace(x_limit[0], x_limit[1], 500)
    # Apply the piecewise logic: C before t0, C + m*(t-t0) after t0
    y_fit = np.where(x_fit < t0_data, C_data, C_data + m_data * (x_fit - t0_data))
    
    # Plot the fit as a dashed line. (Added to legend automatically via the label argument)
    if PLOT_SKYSHINE_FIT:
        ax.plot(x_fit, y_fit, color='magenta', linestyle='--', linewidth=2.5)
        ax.text(0.15, 0.7, r"$\chi^2 / \text{dof} = $" + f"{chi2:.1f}/{ndof}", transform=ax.transAxes)
    ### END NEW
      
    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=16)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    #ax2.set_ylim([-1.2, 1.2]) # Centered on 0
    ax2.set_ylim([-1.2, 5]) # Centered on 0
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)
    
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.text(0.17, 1.02, 'Preliminary', transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes)

    plt.tight_layout()
    if SAVE_PLOT:
        plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()
    
    
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *

print('done')

### Calculate the appropriate skyshine rate

In [ ]:
# Timing plot

# Standard BNB bunch spacing is ~18.936 ns
# 8 bunches per bin * 10 bins = 80 bunches
bin_width = 18.936 * 10  # Buckets of 16 bunches
bins = np.arange(0, 18.936 * 80 + bin_width, bin_width)

#bins = np.arange(0, 18.936 * 25 + bin_width, bin_width)

# Reorder MC for the stack
mc_stack = [bunches_80_mc[5], bunches_80_mc[4], bunches_80_mc[2], 
            bunches_80_mc[1], bunches_80_mc[3], bunches_80_mc[0]]

common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3
    }

plot_sideband_comparison(
    bins=bins,
    mc_data=mc_stack,
    on_beam_data=bunches_80_data,
    off_beam_data=bunches_80_ext,
    axis_label='Time since first bunch [ns]',
    y_axis_label='Events / 10 BNB bunches',
    x_limit=[0, 18.936 * 80],
    y_limit=[0,400],
    xaxis_minor=50,
    plot_name='../plots/sideband/post_graduation/sideband_time_since_first_bunch_skyshine_fit_10.png',
    DEBUG=True,
    ALPHA_SCALE=0.18,
    CALCULATE_SKYSHINE=True,
    PLOT_SKYSHINE_FIT=True,
    SAVE_PLOT=True,
    **common_args
)

print('done')

### Plot PMT observables

With the skyshine fit in hand, we can populate the PMT distributions with the expected skyshine counts and plot the final sideband observables.

Please re-run the selection scripts with the 1/3 spill cut (bunch time < centers[25])

In [ ]:
def plot_with_skyshine(bins, mc_data, on_beam_data, off_beam_data, skyshine_data, 
                       pot_mc, pot_data, trig_ext, trig_data, expected_skyshine_yield,
                       axis_label, y_axis_label, x_limit, y_limit, xaxis_minor, 
                       plot_name, legend_columns=3,
                       DEBUG=False, ALPHA_SCALE=0.18, SAVE_PLOT=False):
    
    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    
    ### NEW: Calculate the weight required to force the skyshine MC to exactly match your expected yield
    if len(skyshine_data) > 0:
        f_sky = expected_skyshine_yield / len(skyshine_data)
    else:
        f_sky = 0.0
    
    # 2. Prepare Prediction Stack (MC + Skyshine + Off-Beam)
    ### NEW: We inject Skyshine right after External (MC) [which is index 0]
    prediction_data = [mc_data[0], skyshine_data] + mc_data[1:] + [off_beam_data]
    
    ### NEW: We build the weights array to match the new 8-element prediction_data order
    weights = [np.ones(len(mc_data[0])) * f_pot, 
               np.ones(len(skyshine_data)) * f_sky] + \
              [np.ones(len(arr)) * f_pot for arr in mc_data[1:]] + \
              [np.ones(len(off_beam_data)) * f_ext]
    
    # 3. Aesthetics
    ### NEW: Added Skyshine to labels and colors and increased outlines to 7
    labels_base = ['External (MC)', 'Skyshine', 'Out-of-FV', 'CC', 'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors = ['papayawhip', 'darkmagenta', 'lime', 'darkred', 'cyan', 'darkorange', '#0066ff', 'gray']
    color_outline = ['navy'] * 7 + ['black']
    
    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * ALPHA_SCALE
            if DEBUG:
                print(f"Applied {ALPHA_SCALE} scale to index {i} ({label})")
        ### NEW: Just a helpful printout to confirm the math worked
        if 'Skyshine' in label:
            if DEBUG:
                print(f"Scaled {len(skyshine_data)} raw skyshine events to exactly {expected_skyshine_yield:.1f} events")

    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]
    
    # 4. Math for Residuals and Errors
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths = np.diff(bins)
    
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    
    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))

    # --- Plotting ---
    fig = plt.figure(figsize=(7, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            label=labels, color=colors, linewidth=0)

    # B. The Outline
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            histtype='step', color=color_outline, linewidth=0.8)

    # C. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2, 
                fmt='ko', label='On-Beam Data', markersize=4, capsize=0)

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    
    ratio = data_counts[mask] / pred_counts[mask]
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
    
    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=14)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 1.2]) 
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)
    
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.text(0.17, 1.02, 'Preliminary', transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes)

    plt.tight_layout()
    if SAVE_PLOT:
        plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()
    
print('done')

In [ ]:
path_folder = '../plots/sideband/post_graduation/'

radius_skyshine = np.sqrt(np.array(skyshine_vertex_x)**2 + np.array(skyshine_vertex_z)**2)

common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3
    }

# Expected skyshine from fit within the sideband region
EXPECTED_SKYSHINE = 43.5

# energy
mc_stack = [MC_reco_E[5], MC_reco_E[4], MC_reco_E[2], MC_reco_E[1], MC_reco_E[3], MC_reco_E[0]]
plot_with_skyshine(
    bins=np.arange(5, 13, 1),
    mc_data=mc_stack,
    on_beam_data=reco_E,
    off_beam_data=ext_reco_E,
    skyshine_data=skyshine_reco_E, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed energy [MeV]',
    y_axis_label='Events / 1 MeV bin',
    x_limit=[5, 12],
    y_limit=[0,80],
    xaxis_minor=0.5,
    plot_name=path_folder + 'sideband_reco_energy.png',
    DEBUG=True,
    SAVE_PLOT=False,
    **common_args
)


mc_stack_hits = [MC_hits[5], MC_hits[4], MC_hits[2], MC_hits[1], MC_hits[3], MC_hits[0]]
plot_with_skyshine(
    bins=np.arange(10, 50, 4),
    mc_data=mc_stack_hits,
    on_beam_data=hits,
    off_beam_data=ext_hits,
    skyshine_data=skyshine_hits, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='PMT hits',
    y_axis_label='Events / 4 hit bin',
    x_limit=[10, 50],
    y_limit=[0,100],
    xaxis_minor=1,
    plot_name=path_folder + 'sideband_PMT_hits.png',
    DEBUG=True,
    SAVE_PLOT=False,
    **common_args
)

mc_stack_r = [radius_mc[5], radius_mc[4], radius_mc[2], radius_mc[1], radius_mc[3], radius_mc[0]]
plot_with_skyshine(
    bins=np.arange(0, 0.8, 0.1),
    mc_data=mc_stack_r,
    on_beam_data=radius_data,
    off_beam_data=radius_ext,
    skyshine_data=radius_skyshine, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed distance to center [m]',
    y_axis_label='Events / 10 cm bin',
    x_limit=[0, 0.7],
    y_limit=[0,100],
    xaxis_minor=0.1,
    plot_name=path_folder + 'sideband_radius.png',
    DEBUG=True,
    SAVE_PLOT=False,
    **common_args
)

mc_stack_y = [MC_vertex_y[5], MC_vertex_y[4], MC_vertex_y[2], MC_vertex_y[1], MC_vertex_y[3], MC_vertex_y[0]]
plot_with_skyshine(
    bins=np.arange(-1, 1.25, 0.25),
    mc_data=mc_stack_y,
    on_beam_data=vertex_y,
    off_beam_data=ext_vertex_y,
    skyshine_data=skyshine_vertex_y, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed vertex (Y) [m]',
    y_axis_label='Events / 50 cm bin',
    x_limit=[-1, 1],
    y_limit=[0,100],
    xaxis_minor=0.2,
    plot_name=path_folder + 'sideband_Y.png',
    DEBUG=True,
    SAVE_PLOT=False,
    **common_args
)

vars_cv = [('x', MC_CV_x, CV_x, ext_CV_x, skyshine_CV_x),
           ('y', MC_CV_y, CV_y, ext_CV_y, skyshine_CV_y),
           ('z', MC_CV_z, CV_z, ext_CV_z, skyshine_CV_z)]
for name, mc_list, data_list, ext_list, sky_list in vars_cv:
    mc_stack = [mc_list[5], mc_list[4], mc_list[2], mc_list[1], mc_list[3], mc_list[0]]
    plot_with_skyshine(
        bins=np.arange(-1, 1.25, 0.25) if name != 'z' else np.arange(-1, 0.6, 0.2),
        mc_data=mc_stack,
        on_beam_data=data_list,
        off_beam_data=ext_list,
        skyshine_data=sky_list, # <-- Pass the new skyshine simulated energies here
        expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
        axis_label=r'$C_V$ ' + f'({name})',
        y_axis_label='Events / bin',
        x_limit=[-1, 1] if name != 'z' else [-1, 0.4],
        y_limit=[0,80] if name != 'y' else [0, 100] ,
        xaxis_minor=0.1,
        plot_name=f'{path_folder}sideband_Cv_{name}.png',
        DEBUG=True,
        SAVE_PLOT=False,
        **common_args
    )
    

print('done')

### We can also test the dirt scaling factor against the CCinc sideband

In [ ]:
# simple event rate bin for dirt scaling factor

# ---------------------------------------------------------
# 1D RATE NORMALIZATION (Total Yield Matching)
# ---------------------------------------------------------
def compute_dirt_scale_with_skyshine(
        mc_stack,
        data_events,
        ext_events,
        skyshine_events,
        expected_skyshine_yield,
        pot_mc,
        pot_data,
        trig_ext,
        trig_data,
        verbose=True):

    # ---------------------------------------------------------
    # Scaling factors
    # ---------------------------------------------------------
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext

    # ---------------------------------------------------------
    # Totals
    # ---------------------------------------------------------
    total_data = len(data_events)

    # Fixed MC (everything except dirt = index 0)
    total_fixed_mc = sum(len(mc_stack[i]) * f_pot
                         for i in range(1, len(mc_stack)))

    # Off-beam
    total_off_beam = len(ext_events) * f_ext

    # Skyshine (forced to expected yield, just like your plot code)
    if skyshine_events is not None and len(skyshine_events) > 0:
        total_skyshine = expected_skyshine_yield
    else:
        total_skyshine = 0.0

    total_fixed = total_fixed_mc + total_off_beam + total_skyshine

    # Unscaled dirt MC
    total_ext_mc = len(mc_stack[0]) * f_pot

    # Solve for dirt scale
    if total_ext_mc <= 0:
        rate_scale_factor = 0.0
    else:
        rate_scale_factor = (total_data - total_fixed) / total_ext_mc

    # ---------------------------------------------------------
    # Printout
    # ---------------------------------------------------------
    if verbose:
        print("\n" + "="*45)
        print(" 1D RATE NORMALIZATION (WITH SKYSHINE)")
        print("="*45)
        print(f"Total Data Events:         {total_data}")
        print(f"Fixed MC (no dirt):        {total_fixed_mc:.1f}")
        print(f"Off-beam contribution:     {total_off_beam:.1f}")
        print(f"Skyshine (fixed):          {total_skyshine:.1f}")
        print(f"Total Fixed Backgrounds:   {total_fixed:.1f}")
        print(f"Total Unscaled Ext MC:     {total_ext_mc:.1f}")
        print("-" * 45)
        print(f"Calculated Dirt Scale:     {rate_scale_factor:.4f}")
        print("="*45 + "\n")

    return rate_scale_factor


mc_stack = [bunches_80_mc[5], bunches_80_mc[4], bunches_80_mc[2], 
            bunches_80_mc[1], bunches_80_mc[3], bunches_80_mc[0]]

rate_scale_factor = compute_dirt_scale_with_skyshine(
    mc_stack=mc_stack,
    data_events=bunches_80_data,
    ext_events=bunches_80_ext,
    skyshine_events=skyshine_bunch_times,
    expected_skyshine_yield=43.5,
    pot_mc=POT_MC,
    pot_data=POT_DATA,
    trig_ext=TRIG_EXT,
    trig_data=TRIG_DATA
)

---------------
## 2. Updated Sideband Analysis

Introduce weighting to adjust CC backgrounds and updated fitting machinery to help reduce the skyshine systematic.

### Reload MC with weighted samples

In [ ]:
MC_directory = '../42_44/'
file_names = [f for f in os.listdir(MC_directory) if os.path.isfile(os.path.join(MC_directory, f))]

# POT for World samples --> 5.518e16 POT per file
POT = 2.3e20     # MC POT (modify if needed, calculated based on total number of files present)

# scalar accumulators
MC_cluster_number    = []
MC_cluster_time      = []
MC_cluster_charge    = []
MC_cluster_Qb        = []
MC_cluster_Hits      = []
MC_TankMRDCoinc      = []
MC_NoVeto            = []
MC_recoVtx           = []
MC_recoVty           = []
MC_recoVtz           = []
MC_Vtx               = []
MC_Vty               = []
MC_Vtz               = []
MC_Vtt               = []
MC_bunchTimes        = []
MC_External          = []
MC_FV                = []
MC_NC                = []
MC_QEL               = []
MC_MEC               = []
MC_PDG               = []
MC_Z                 = []
MC_file_index        = []   # which file each cluster came from
MC_event_number      = []

# jagged accumulators
MC_hitX  = []
MC_hitY  = []
MC_hitZ  = []
MC_hitT  = []
MC_hitPE = []
MC_hitID = []
MC_MRD_hitT  = []
MC_FMV_hitT  = []

# CV weights (MicroBooNE tuning)
MC_CV_weight = []

for file_idx in trange(len(file_names), desc='Loading MC files'):
    path = os.path.join(MC_directory, file_names[file_idx])
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    MC_cluster_number.append(T['clusterNumber'].array(library='np'))
    MC_cluster_charge.append(T['clusterPE'].array(library='np'))
    MC_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    MC_cluster_Hits.append(T['clusterHits'].array(library='np'))
    MC_cluster_time.append(T['clusterTime'].array(library='np'))
    MC_bunchTimes.append(T['bunchTimes'].array(library='np'))
    MC_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    MC_NoVeto.append(T['NoVeto'].array(library='np'))
    MC_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    MC_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    MC_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))
    MC_event_number.append(CEN)
    MC_file_index.append(np.full(len(CEN), file_idx, dtype=np.int32))

    MC_hitX.append(T['hitX'].array())
    MC_hitY.append(T['hitY'].array())
    MC_hitZ.append(T['hitZ'].array())
    MC_hitT.append(T['hitT'].array())
    MC_hitPE.append(T['hitPE'].array())
    MC_hitID.append(T['hitDetID'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']
    vx  = S['trueNuIntxVtx_X'].array(library='np')
    vy  = S['trueNuIntxVtx_Y'].array(library='np')
    vz  = S['trueNuIntxVtx_Z'].array(library='np')

    vx_cm = vx / 100
    vy_cm = vy / 100 + 0.1446
    vz_cm = vz / 100 - 1.681
    r2    = vx_cm**2 + vz_cm**2
    ext_flags = (r2 > radius**2)    | (np.abs(vy_cm) > half_height)
    fv_flags  = (r2 < FV_radius**2) & (np.abs(vy_cm) < FV_half_height)

    # map trigger-level quantities onto clusters via CEN
    MC_Vtx.append(vx_cm[CEN])
    MC_Vty.append(vy_cm[CEN])
    MC_Vtz.append(vz_cm[CEN])
    MC_Vtt.append(S['trueNuIntxVtx_T'].array(library='np')[CEN])
    MC_External.append(ext_flags[CEN])
    MC_FV.append(fv_flags[CEN])
    MC_NC.append(S['trueNC'].array(library='np')[CEN])
    MC_QEL.append(S['trueQEL'].array(library='np')[CEN])
    MC_MEC.append(S['trueMEC'].array(library='np')[CEN])
    MC_PDG.append(S['trueFSLPdg'].array(library='np')[CEN])
    MC_Z.append(S['trueTargetZ'].array(library='np')[CEN])

    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    MC_MRD_hitT.append(mrd_t[CEN])
    MC_FMV_hitT.append(veto_t[CEN])
    
    MC_CV_weight.append(S['weight_TunedCentralValue_UBGenie'].array(library='np')[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

MC_cluster_number  = np.concatenate(MC_cluster_number)
MC_cluster_time    = np.concatenate(MC_cluster_time)
MC_cluster_charge  = np.concatenate(MC_cluster_charge)
MC_cluster_Qb      = np.concatenate(MC_cluster_Qb)
MC_cluster_Hits    = np.concatenate(MC_cluster_Hits)
MC_bunchTimes      = np.concatenate(MC_bunchTimes)
MC_TankMRDCoinc    = np.concatenate(MC_TankMRDCoinc)
MC_NoVeto          = np.concatenate(MC_NoVeto)
MC_recoVtx         = np.concatenate(MC_recoVtx)
MC_recoVty         = np.concatenate(MC_recoVty)
MC_recoVtz         = np.concatenate(MC_recoVtz)
MC_event_number    = np.concatenate(MC_event_number)
MC_file_index      = np.concatenate(MC_file_index)
MC_Vtx             = np.concatenate(MC_Vtx)
MC_Vty             = np.concatenate(MC_Vty)
MC_Vtz             = np.concatenate(MC_Vtz)
MC_Vtt             = np.concatenate(MC_Vtt)
MC_External        = np.concatenate(MC_External)
MC_FV              = np.concatenate(MC_FV)
MC_NC              = np.concatenate(MC_NC)
MC_QEL             = np.concatenate(MC_QEL)
MC_MEC             = np.concatenate(MC_MEC)
MC_PDG             = np.concatenate(MC_PDG)
MC_Z               = np.concatenate(MC_Z)

MC_hitX     = ak.concatenate(MC_hitX)
MC_hitY     = ak.concatenate(MC_hitY)
MC_hitZ     = ak.concatenate(MC_hitZ)
MC_hitT     = ak.concatenate(MC_hitT)
MC_hitPE    = ak.concatenate(MC_hitPE)
MC_hitID    = ak.concatenate(MC_hitID)
MC_MRD_hitT = ak.concatenate(MC_MRD_hitT)
MC_FMV_hitT = ak.concatenate(MC_FMV_hitT)

MC_CV_weight = np.concatenate(MC_CV_weight)

print(f'POT: {POT:.3e}')
print(f'Total clusters: {len(MC_cluster_number)}')
print('\ndone')

### You can use the same on and off beam cells as in the Legacy code

### Load MC skyshine --> here we can utilize more samples

In [ ]:
file_names = [
    '../skyshine_100k_no_ext_clusters.root',
    '../skyshine_100k_no_ext_clusters_2.root',
    '../skyshine_100k_no_ext_clusters_3.root'
]

# scalar accumulators
skyshine_cluster_number    = []
skyshine_cluster_time      = []
skyshine_cluster_charge    = []
skyshine_cluster_Qb        = []
skyshine_cluster_Hits      = []
skyshine_TankMRDCoinc      = []
skyshine_NoVeto            = []
skyshine_recoVtx           = []
skyshine_recoVty           = []
skyshine_recoVtz           = []
skyshine_event_number      = []

# jagged accumulators
skyshine_hitX  = []
skyshine_hitY  = []
skyshine_hitZ  = []
skyshine_hitT  = []
skyshine_hitPE = []
skyshine_hitID = []
skyshine_MRD_hitT  = []
skyshine_FMV_hitT  = []

for file_name in file_names:
    path = os.path.join(file_name)
    root = uproot.open(path)

    # ── cluster tree ──────────────────────────────────────────────────────────
    T   = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array(library='np')

    skyshine_cluster_number.append(T['clusterNumber'].array(library='np'))
    skyshine_cluster_charge.append(T['clusterPE'].array(library='np'))
    skyshine_cluster_Qb.append(T['clusterChargeBalance'].array(library='np'))
    skyshine_cluster_Hits.append(T['clusterHits'].array(library='np'))
    skyshine_cluster_time.append(T['clusterTime'].array(library='np'))
    skyshine_TankMRDCoinc.append(T['TankMRDCoinc'].array(library='np'))
    skyshine_NoVeto.append(T['NoVeto'].array(library='np'))
    skyshine_recoVtx.append(T['recoLeastSqVtxX'].array(library='np'))
    skyshine_recoVty.append(T['recoLeastSqVtxY'].array(library='np'))
    skyshine_recoVtz.append(T['recoLeastSqVtxZ'].array(library='np'))
    skyshine_event_number.append(CEN)

    skyshine_hitX.append(T['hitX'].array())
    skyshine_hitY.append(T['hitY'].array())
    skyshine_hitZ.append(T['hitZ'].array())
    skyshine_hitT.append(T['hitT'].array())
    skyshine_hitPE.append(T['hitPE'].array())
    skyshine_hitID.append(T['hitDetID'].array())

    # ── trigger tree ──────────────────────────────────────────────────────────
    S   = root['phaseIITriggerTree']

    # map trigger-level quantities onto clusters via CEN
    mrd_t   = S['MRDhitT'].array()
    veto_t  = S['FMVhitT'].array()
    skyshine_MRD_hitT.append(mrd_t[CEN])
    skyshine_FMV_hitT.append(veto_t[CEN])

# ── concatenate everything once ───────────────────────────────────────────────
print('\nManaging arrays...\n')

skyshine_cluster_number  = np.concatenate(skyshine_cluster_number)
skyshine_cluster_time    = np.concatenate(skyshine_cluster_time)
skyshine_cluster_charge  = np.concatenate(skyshine_cluster_charge)
skyshine_cluster_Qb      = np.concatenate(skyshine_cluster_Qb)
skyshine_cluster_Hits    = np.concatenate(skyshine_cluster_Hits)
skyshine_TankMRDCoinc    = np.concatenate(skyshine_TankMRDCoinc)
skyshine_NoVeto          = np.concatenate(skyshine_NoVeto)
skyshine_recoVtx         = np.concatenate(skyshine_recoVtx)
skyshine_recoVty         = np.concatenate(skyshine_recoVty)
skyshine_recoVtz         = np.concatenate(skyshine_recoVtz)
skyshine_event_number    = np.concatenate(skyshine_event_number)

skyshine_hitX     = ak.concatenate(skyshine_hitX)
skyshine_hitY     = ak.concatenate(skyshine_hitY)
skyshine_hitZ     = ak.concatenate(skyshine_hitZ)
skyshine_hitT     = ak.concatenate(skyshine_hitT)
skyshine_hitPE    = ak.concatenate(skyshine_hitPE)
skyshine_hitID    = ak.concatenate(skyshine_hitID)
skyshine_MRD_hitT = ak.concatenate(skyshine_MRD_hitT)
skyshine_FMV_hitT = ak.concatenate(skyshine_FMV_hitT)

print(f'Total clusters: {len(skyshine_cluster_number)}')
print('\ndone')

### Apply selection cuts

In [ ]:
# First, MC

# the dimensions of the saved arrays are as follows:
# [0]: nuNCQE (signal)
# [1]: NCother
# [2]: CC
# [3]: nubarNCQE
# [4]: Out-of-FV (OOFV)
# [5]: External (interactions outside of the water tank entirely)

MC_reco_E = [[], [], [], [], [], []]
MC_CV_x = [[], [], [], [], [], []]
MC_CV_y = [[], [], [], [], [], []]
MC_CV_z = [[], [], [], [], [], []]
MC_bunch_times = [[], [], [], [], [], []]
MC_vertex_x = [[], [], [], [], [], []]
MC_vertex_y = [[], [], [], [], [], []]
MC_vertex_z = [[], [], [], [], [], []]
MC_hits = [[], [], [], [], [], []]

MC_weights_by_cat = [[], [], [], [], [], []]


for i in trange(len(MC_cluster_time), desc = 'applying NCQE selection cuts to MC...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if MC_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if MC_bunchTimes[i] > bunch_time_cutoff + time_to_prompt_end:
        continue
    # minimum 10 hits
    if MC_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    # No MRD/FMV correlated activity
    if (
        MC_TankMRDCoinc[i] == 1
        or MC_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in MC_MRD_hitT[i])
        or any(0 <= t <= 300 for t in MC_FMV_hitT[i])
    ):
        continue
        
    # Cluster time within 1.6 us beam spill
    if MC_bunchTimes[i] > bunch_time_cutoff:
        continue
        
    # Add earlier time cut (reject anything later than the 25th bunch center)
    # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
    if MC_bunchTimes[i] > centers[25]:
        continue
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(MC_cluster_time)):        # look at the subsequent clusters in this event
        if MC_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if MC_bunchTimes[k] < bunch_time_cutoff + time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if MC_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut (FV defined in first cell)
    if MC_recoVtx[i] < -50:
        continue
    r2 = MC_recoVtx[i]**2 + MC_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(MC_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(MC_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    # INVERT BUNCH CUT FOR SIDEBAND
    if bunch_cutting(MC_bunchTimes[i], centers) == True:
        continue
        
    cw = pairwise_relative_direction(MC_hitX[i], MC_hitY[i], MC_hitZ[i], MC_hitPE[i], MC_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
        
    # event categorization
    if MC_External[i] == True:
        indy = 5
        MC_reco_E[indy].append(recoE)
        MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
        MC_bunch_times[indy].append(MC_bunchTimes[i])
        MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
        MC_hits[indy].append(MC_cluster_Hits[i])
        MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
    else:
        if MC_FV[i] == True:
            if MC_NC[i] == 1:       # NC
                if MC_QEL[i] == 1:
                    if MC_Z[i] == 8:
                        if MC_PDG[i] == 14 or MC_PDG[i] == 12:      # nu (mu + e)
                            # nuNCQE
                            indy = 0
                            MC_reco_E[indy].append(recoE)
                            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                            MC_bunch_times[indy].append(MC_bunchTimes[i])
                            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                            MC_hits[indy].append(MC_cluster_Hits[i])
                            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                        elif MC_PDG[i] == -14 or MC_PDG[i] == -12:
                            # nubarNCQE
                            indy = 3
                            MC_reco_E[indy].append(recoE)
                            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                            MC_bunch_times[indy].append(MC_bunchTimes[i])
                            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                            MC_hits[indy].append(MC_cluster_Hits[i])
                            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                    else:
                        # NC elastic on H (NCother)
                        indy = 1
                        MC_reco_E[indy].append(recoE)
                        MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                        MC_bunch_times[indy].append(MC_bunchTimes[i])
                        MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                        MC_hits[indy].append(MC_cluster_Hits[i])
                        MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
                else:
                    # NC, not QEL (NCother)
                    indy = 1
                    MC_reco_E[indy].append(recoE)
                    MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                    MC_bunch_times[indy].append(MC_bunchTimes[i])
                    MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                    MC_hits[indy].append(MC_cluster_Hits[i])
                    MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
            else:
                # CC
                indy = 2
                MC_reco_E[indy].append(recoE)
                MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
                MC_bunch_times[indy].append(MC_bunchTimes[i])
                MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
                MC_hits[indy].append(MC_cluster_Hits[i])
                MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
        else:
            # out-of-FV
            indy = 4
            MC_reco_E[indy].append(recoE)
            MC_CV_x[indy].append(cw[0]); MC_CV_y[indy].append(cw[1]); MC_CV_z[indy].append(cw[2])
            MC_bunch_times[indy].append(MC_bunchTimes[i])
            MC_vertex_x[indy].append(MC_recoVtx[i]); MC_vertex_y[indy].append(MC_recoVty[i]); MC_vertex_z[indy].append(MC_recoVtz[i])
            MC_hits[indy].append(MC_cluster_Hits[i])
            MC_weights_by_cat[indy].append(MC_CV_weight[i][0])
          

print('done')

### Retain the same data selection, but off-beam can be more strategic

Rather than nuking the statistics with the 2 sigma bunch cut for the off-beam data, we can instead drop the bunch cuts and scale the off-beam exposure accordingly.

In [ ]:
# Apply selection cuts for On-Beam data

reco_E = []
CV_x = []
CV_y = []
CV_z = []
bunch_times = []
vertex_x = []
vertex_y = []
vertex_z = []
hits = []
reco_runs = []


for i in trange(len(cluster_time), desc = 'applying NCQE selection cuts to on-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc[i] == 1 or NoVeto[i] == 0 or MRD_activity[i] != 0 or Veto_activity[i] != 0:
        continue
        
    if runs[i] > 4000:   # FY23
        if cluster_time[i] < spill_start_23:
            continue
        if cluster_time[i] > spill_end_23:
            continue
            
        # add for earlier time
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
        if cluster_time[i] > centers_data_23[25]:
            continue
        
        # INVERT BUNCH CUT FOR SIDEBAND
        if bunch_cutting(cluster_time[i], centers_data_23) == True:
            continue
    else:
        if cluster_time[i] < spill_start_22:
            continue
        if cluster_time[i] > spill_end_22:
            continue
            
        # add for earlier time
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
        if cluster_time[i] > centers_data_22[25]:
            continue
        
        # INVERT BUNCH CUT FOR SIDEBAND
        if bunch_cutting(cluster_time[i], centers_data_22) == True:
            continue
        
    # CCB cut
    if cluster_Qb[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco[i] < -50:
        continue
    r2 = Vtx_reco[i]**2 + Vtz_reco[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX[i], hitY[i], hitZ[i], hitPE[i], hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    # event selection
    reco_E.append(recoE)
    CV_x.append(cw[0]); CV_y.append(cw[1]); CV_z.append(cw[2])
    bunch_times.append(cluster_time[i])
    vertex_x.append(Vtx_reco[i]); vertex_y.append(Vty_reco[i]); vertex_z.append(Vtz_reco[i])
    hits.append(cluster_Hits[i])
    reco_runs.append(runs[i])
          

print('done')

In [ ]:
ext_reco_E = []
ext_CV_x = []
ext_CV_y = []
ext_CV_z = []
ext_bunch_times = []
ext_vertex_x = []
ext_vertex_y = []
ext_vertex_z = []
ext_hits = []
ext_reco_runs = []

for i in trange(len(cluster_time_ext), desc = 'applying NCQE selection cuts to off-beam data...'):
    
    # ------ Has cluster ------- #
    # extracted ntuples have been filtered to only include clusters >= 10 hits
    # only_prompt_cluster covers CN == 0 and the cluster is in the prompt window
    if only_prompt_cluster_ext[i] == False:
        continue
    # ------ Has cluster ------- #
    
    # No FMV or MRD activity
    if TankMRDCoinc_ext[i] == 1 or NoVeto_ext[i] == 0 or MRD_activity_ext[i] != 0 or Veto_activity_ext[i] != 0:
        continue
        
    if runs_ext[i] > 4000:   # FY23
        if cluster_time_ext[i] < spill_start_23:
            continue
        if cluster_time_ext[i] > spill_end_23:
            continue
            
        # add for earlier time
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
        if cluster_time_ext[i] > centers_data_23[25]:
            continue
        
        # skip bunch cutting, scale the exposure
        if bunch_cutting(cluster_time_ext[i], centers_data_23) == True:
            continue
        
    else:
        if cluster_time_ext[i] < spill_start_22:
            continue
        if cluster_time_ext[i] > spill_end_22:
            continue
            
        # add for earlier time
        # AS A FIRST PASS PRIOR TO SKYSHINE RATE CALCULATION, DISABLE THIS
        if cluster_time_ext[i] > centers_data_22[25]:
            continue
        
        # skip bunch cutting, scale the exposure
        if bunch_cutting(cluster_time_ext[i], centers_data_22) == True:
            continue
        
    # CCB cut
    if cluster_Qb_ext[i] >= 0.6:
        continue
        
    # reconstructed FV cut
    if Vtx_reco_ext[i] < -50:
        continue
    r2 = Vtx_reco_ext[i]**2 + Vtz_reco_ext[i]**2
    if r2 > FV_radius**2 or np.abs(Vty_reco_ext[i]) > FV_half_height:
        continue
        
    # Energy reco cut
    recoE = reco_energy(cluster_charge_ext[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(hitX_ext[i], hitY_ext[i], hitZ_ext[i], hitPE_ext[i], hitT_ext[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
        
    # event selection
    ext_reco_E.append(recoE)
    ext_CV_x.append(cw[0]); ext_CV_y.append(cw[1]); ext_CV_z.append(cw[2])
    ext_bunch_times.append(cluster_time_ext[i])
    ext_vertex_x.append(Vtx_reco_ext[i]); ext_vertex_y.append(Vty_reco_ext[i]); ext_vertex_z.append(Vtz_reco_ext[i])
    ext_hits.append(cluster_Hits_ext[i])
    ext_reco_runs.append(runs_ext[i])
          

print('done')

In [ ]:
skyshine_reco_E = []
skyshine_CV_x = []
skyshine_CV_y = []
skyshine_CV_z = []
skyshine_bunch_times = []
skyshine_vertex_x = []
skyshine_vertex_y = []
skyshine_vertex_z = []
skyshine_hits = []


# do not apply timing cuts
for i in trange(len(skyshine_cluster_time), desc = 'applying NCQE selection cuts to simulated skyshine...'):
    
    # ------ Has cluster ------- #
    # primary cluster
    if skyshine_cluster_number[i] != 0:
        continue
    # cluster is within the prompt window
    if skyshine_cluster_time[i] > bunch_time_cutoff + time_to_prompt_end:
        continue
    # minimum 10 hits
    if skyshine_cluster_Hits[i] < 10:
        continue
    # ------ Has cluster ------- #
        
    if (
        skyshine_TankMRDCoinc[i] == 1
        or skyshine_NoVeto[i] == 0
        or any(0 <= t <= 300 for t in skyshine_MRD_hitT[i])
        or any(0 <= t <= 300 for t in skyshine_FMV_hitT[i])
    ):
        continue
        
    # Cluster time within 1.6 us beam spill of particle creation (t0 = 0 for these simulations)
    if skyshine_cluster_time[i] > bunch_time_cutoff:
        continue
        
    # No bunch cutting applied
        
    # Only one prompt cluster
    skip_event = False
    for k in range(i+1,len(skyshine_cluster_time)):        # look at the subsequent clusters in this event
        if skyshine_cluster_number[k] != 0:   # if the next cluster is a secondary cluster
            # check if the time of the next cluster is within the prompt 2us window
            if skyshine_cluster_time[k] < bunch_time_cutoff + time_to_prompt_end:
                skip_event = True
                break
        else:     # we found an event without a secondary cluster
            break
    if skip_event:
        continue  # Skip the entire event
        
    # 6. CCB cut
    if skyshine_cluster_Qb[i] >= 0.6:
        continue
        
    # 8. reconstructed FV cut
    if skyshine_recoVtx[i] < -50:
        continue
    r2 = skyshine_recoVtx[i]**2 + skyshine_recoVtz[i]**2
    if r2 > FV_radius**2 or np.abs(skyshine_recoVty[i]) > FV_half_height:
        continue
        
    # 7. Energy reco cut
    recoE = reco_energy(skyshine_cluster_charge[i])
    if recoE < 5 or recoE > 12:
        continue
        
    cw = pairwise_relative_direction(skyshine_hitX[i], skyshine_hitY[i], skyshine_hitZ[i], skyshine_hitPE[i], skyshine_hitT[i])
    if is_inside_rotated_parabola(cw[2], cw[0]) == False:
        continue
        
    skyshine_reco_E.append(recoE)
    skyshine_CV_x.append(cw[0]); skyshine_CV_y.append(cw[1]); skyshine_CV_z.append(cw[2])
    skyshine_bunch_times.append(skyshine_cluster_time[i])
    skyshine_vertex_x.append(skyshine_recoVtx[i]); 
    skyshine_vertex_y.append(skyshine_recoVty[i]);
    skyshine_vertex_z.append(skyshine_recoVtz[i])
    skyshine_hits.append(skyshine_cluster_Hits[i])
    

print('done')

In [ ]:
# 1. Scaling factors
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
POT_MC = POT/(1e20)
POT_DATA = total_POT_TOR860
TRIG_DATA = total_triggers
TRIG_EXT = total_ext_triggers

# Spill window lengths: 1533 (2022) and 1530 (2023) --> this is the exposure for the off-beam data
# The sideband was the inter-bunch regions for a total of 80 intervals
# The bunches are spaced by 18.936 ns, therefore the sideband only explores 18.936/2 - 3.59*2 = 2.288ns per “half” 
# —> multiply by 2 because we’re exploring the space between the bunches = 4.576 x 80 bunches = 366ns (full spill)

#TRIG_EXT = TRIG_EXT/(366/1530)   # off-beam data has ~4-5x the exposure than the bunch cutting

# only thing to be concerned with is the off-beam vs bunch binning may be wonky with low stats

# 2. Chain all bunch times to a common t0
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
def get_normalized_times(times, runs, weights=None, mc=False):

    norm_times = []
    norm_weights = []

    for i, (t, r) in enumerate(zip(times, runs)):

        if mc:
            t0 = centers[0]
        else:
            t0 = centers_data_22[0] if r < 4000 else centers_data_23[0]

        tnorm = t - t0

        # Filter to only include the 80 bunches
        if 0 <= tnorm < (80 * 18.936):

            norm_times.append(tnorm)

            if weights is not None:
                norm_weights.append(weights[i])

    if weights is not None:
        return np.array(norm_times), np.array(norm_weights)

    return np.array(norm_times)


# 1. Process MC (Categories 0-5)
bunches_80_mc = []
bunches_80_weights = []

for i in range(6):

    t, w = get_normalized_times(
        MC_bunch_times[i],
        [0]*len(MC_bunch_times[i]),
        weights=MC_weights_by_cat[i],
        mc=True
    )

    bunches_80_mc.append(t)
    bunches_80_weights.append(w)

# 2. Process On-Beam Data
bunches_80_data = get_normalized_times(bunch_times, reco_runs)

# 3. Process Off-Beam (External)
bunches_80_ext = get_normalized_times(ext_bunch_times, ext_reco_runs)


# 3. Calculation of radial distance of the reco vertex
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
# For MC (6 categories)
radius_mc = [np.sqrt(np.array(MC_vertex_x[i])**2 + np.array(MC_vertex_z[i])**2) for i in range(6)]
# For On-Beam Data
radius_data = np.sqrt(np.array(vertex_x)**2 + np.array(vertex_z)**2)
# For Off-Beam (EXT) Data
radius_ext = np.sqrt(np.array(ext_vertex_x)**2 + np.array(ext_vertex_z)**2)


# 4. Relative time from nearest bunch center
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *
def distance_to_bunch_center(cluster_time, centers):
    avg_diff = 0.0588
    # Ensure 'centers' is defined in your environment
    shifted_centers = centers - avg_diff 
    return np.min(np.abs(shifted_centers - cluster_time))

# --- MC Calculation (Static) ---
# Assuming 'centers' is your MC bunch center array
delta_t_mc = [
    np.array([distance_to_bunch_center(t, centers) for t in cat]) 
    for cat in MC_bunch_times
]

# --- On-Beam Data (Run Dependent) ---
delta_t_data_list = []
for t, run in zip(bunch_times, reco_runs):
    # Select epoch centers based on run number
    current_centers = centers_data_22 if run < 4000 else centers_data_23
    delta_t_data_list.append(distance_to_bunch_center(t, current_centers))

delta_t_data = np.array(delta_t_data_list)

# --- Off-Beam Data (Run Dependent) ---
delta_t_ext_list = []
for t, run in zip(ext_bunch_times, ext_reco_runs):
    # Same logic for Off-beam/External data
    current_centers = centers_data_22 if run < 4000 else centers_data_23
    delta_t_ext_list.append(distance_to_bunch_center(t, current_centers))

delta_t_ext = np.array(delta_t_ext_list)
    
    
# * * * * * * * * * * * * * * * * * * * * * * * * * * * * * * *

print('done')

In [ ]:
"""
skyshine fitting machinery
────────────────────────────────────────────────────────────────────────────────
Profile-likelihood skyshine extraction for the ANNIE BNB sideband timing fit.

Three public functions
    profile_skyshine(...)           → analysis engine, returns result dict
    plot_profile_diagnostics(...)   → chi2 / N_signal diagnostic figure
    plot_sideband_comparison(...)   → main stacked-histogram + residual figure
                                      calls the two functions above internally
                                      returns the profile result dict

Usage (minimal)
────────────────
    result = plot_sideband_comparison(
        bins, mc_stack, bunches_80_data, bunches_80_ext,
        pot_mc=POT_MC, pot_data=POT_DATA,
        trig_ext=TRIG_EXT, trig_data=TRIG_DATA,
        axis_label='Time since first bunch [ns]',
        y_axis_label='Events / 4 BNB bunches',
        x_limit=[0, 18.936*80], y_limit=[0, 200], xaxis_minor=50,
        plot_name='plots/sideband.png',
        CALCULATE_SKYSHINE=True, PLOT_SKYSHINE_FIT=True, PLOT_PROFILE=True,
        t_cut=t_cut, EXPOSURE_SCALE=1.56,
        DEBUG=True,
    )
    # result['N_signal'], result['err_N_signal'], etc.
"""

from matplotlib.ticker import MultipleLocator


# ══════════════════════════════════════════════════════════════════════════════
# 1.  PROFILE LIKELIHOOD ENGINE
# ══════════════════════════════════════════════════════════════════════════════

def profile_skyshine(bins, data_counts, data_err,
                     pred_counts, pred_err,
                     t_cut, EXPOSURE_SCALE,
                     t0_grid=None, DEBUG=False):
    """
    Profile likelihood scan over t0 for the piecewise-ramp skyshine model

        f(t; C, m, t0) = C + m * max(t − t0, 0)

    For each fixed t0 the model is linear in (C, m), so the WLS solution and
    its exact covariance matrix are analytic.  Scanning t0 on a dense grid
    avoids the discontinuous-gradient problem that makes curve_fit + pcov
    unreliable for piecewise functions.

    Parameters
    ----------
    bins         : array (n_bins+1,)  uniform bin edges [ns]
    data_counts  : array (n_bins,)    on-beam counts per bin
    data_err     : array (n_bins,)    on-beam errors  (√N, floored at 1)
    pred_counts  : array (n_bins,)    stacked MC+EXT prediction
    pred_err     : array (n_bins,)    prediction statistical error (√(Σw²))
    t_cut        : float              upper edge of 1/3-spill signal window [ns]
    EXPOSURE_SCALE : float            sideband → signal band livetime factor
    t0_grid      : array | None       grid of t0 values; default linspace(50, t_cut−20, 600)
    DEBUG        : bool               print fit summary to stdout

    Returns
    -------
    dict with keys
        t0_best, t0_err         best-fit t0 [ns] and (err_lo, err_hi) from Δχ²=1
        C_best, m_data_best     WLS parameters at best-fit t0
        m_pred_best, rate_best  excess slope and rate [events ns⁻²]
        chi2_min, ndof          goodness of fit at t0_best
        N_sideband, err_N       sideband estimate and (err_lo, err_hi)
        N_signal, err_N_signal  signal-band estimate and (err_lo, err_hi)
        t0_grid                 the scanned t0 values
        chi2_profile            χ²(t0) for data
        N_profile               N_sideband(t0)  (NaN where t0 ≥ t_cut)
        in_sigma                boolean mask: Δχ² < 1
    """
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_width   = float(bins[1] - bins[0])          # uniform spacing assumed

    if t0_grid is None:
        t0_grid = np.linspace(50.0, t_cut - 20.0, 600)

    n             = len(t0_grid)
    chi2_data     = np.full(n, np.inf)
    chi2_pred     = np.full(n, np.inf)
    C_data_arr    = np.zeros(n)
    m_data_arr    = np.zeros(n)
    m_pred_arr    = np.zeros(n)
    var_m_arr     = np.zeros(n)     # Var(m_data) from WLS covariance

    for i, t0 in enumerate(t0_grid):
        # design matrix  X[:, 0] = 1,  X[:, 1] = max(t_j − t0, 0)
        ramp = np.maximum(bin_centers - t0, 0.0)
        X    = np.column_stack([np.ones(len(bin_centers)), ramp])

        # ── data WLS ─────────────────────────────────────────────────────────
        W_d  = 1.0 / data_err**2
        XtW  = X.T * W_d                 # (2, n_bins) — each row scaled by weight
        A_d  = XtW @ X                   # (2, 2) normal-equations matrix
        b_d  = XtW @ data_counts
        try:
            cov_d        = np.linalg.inv(A_d)
            beta_d       = cov_d @ b_d
            resid_d      = (data_counts - X @ beta_d) / data_err
            chi2_data[i]  = float(resid_d @ resid_d)
            C_data_arr[i] = beta_d[0]
            m_data_arr[i] = beta_d[1]
            var_m_arr[i]  = cov_d[1, 1]
        except np.linalg.LinAlgError:
            continue

        # ── prediction WLS ───────────────────────────────────────────────────
        W_p   = 1.0 / pred_err**2
        XtW_p = X.T * W_p
        A_p   = XtW_p @ X
        b_p   = XtW_p @ pred_counts
        try:
            cov_p         = np.linalg.inv(A_p)
            beta_p        = cov_p @ b_p
            resid_p       = (pred_counts - X @ beta_p) / pred_err
            chi2_pred[i]  = float(resid_p @ resid_p)
            m_pred_arr[i] = beta_p[1]
        except np.linalg.LinAlgError:
            continue

    # ── best-fit t0 ──────────────────────────────────────────────────────────
    best_idx    = int(np.argmin(chi2_data))
    t0_best     = float(t0_grid[best_idx])
    chi2_min    = float(chi2_data[best_idx])
    C_best      = float(C_data_arr[best_idx])
    m_data_best = float(m_data_arr[best_idx])
    m_pred_best = float(m_pred_arr[best_idx])
    ndof        = len(bin_centers) - 2      # 2 free params (C, m) at fixed t0

    # ── 1-sigma interval  Δχ² < 1 ────────────────────────────────────────────
    in_sigma  = chi2_data < chi2_min + 1.0
    if np.any(in_sigma):
        t0_lo = float(t0_grid[in_sigma][0])
        t0_hi = float(t0_grid[in_sigma][-1])
    else:                                   # degenerate — shouldn't happen
        t0_lo = t0_hi = t0_best
    t0_err_lo = t0_best - t0_lo
    t0_err_hi = t0_hi   - t0_best

    # ── N_sideband profile ───────────────────────────────────────────────────
    delta_t_arr = t_cut - t0_grid
    rate_arr    = (m_data_arr - m_pred_arr) / bin_width
    N_arr       = np.where(delta_t_arr > 0.0,
                           0.5 * rate_arr * delta_t_arr**2,
                           np.nan)

    # ── best-fit N ───────────────────────────────────────────────────────────
    rate_best = (m_data_best - m_pred_best) / bin_width
    delta_t   = t_cut - t0_best
    N_best    = 0.5 * rate_best * delta_t**2

    # slope contribution (t0 fixed at best value, exact WLS covariance)
    dN_dm        = 0.5 * delta_t**2 / bin_width
    err_N_slope  = dN_dm * float(np.sqrt(var_m_arr[best_idx]))

    # t0 contribution: read off N at the Δχ²=1 boundaries
    N_at_lo     = float(N_arr[in_sigma][0])  if np.any(in_sigma) else N_best
    N_at_hi     = float(N_arr[in_sigma][-1]) if np.any(in_sigma) else N_best
    err_N_t0_lo = abs(N_best - N_at_lo)
    err_N_t0_hi = abs(N_best - N_at_hi)

    # combined asymmetric errors (added in quadrature)
    err_N_lo = float(np.sqrt(err_N_slope**2 + err_N_t0_lo**2))
    err_N_hi = float(np.sqrt(err_N_slope**2 + err_N_t0_hi**2))

    if DEBUG:
        _flo = 100 * err_N_lo / N_best if N_best > 0 else float('nan')
        _fhi = 100 * err_N_hi / N_best if N_best > 0 else float('nan')
        print(f"\n{'='*56}")
        print(f"  PROFILE LIKELIHOOD SKYSHINE EXTRACTION")
        print(f"{'='*56}")
        print(f"  t0 (best-fit) :  {t0_best:.1f}  +{t0_err_hi:.1f} / −{t0_err_lo:.1f}  ns")
        print(f"  chi2_min/ndof :  {chi2_min:.2f}/{ndof} = {chi2_min/ndof:.3f}")
        print(f"  C (baseline)  :  {C_best:.2f}  events/bin")
        print(f"  m_data        :  {m_data_best:.5f} ± {np.sqrt(var_m_arr[best_idx]):.5f}  (evts/bin)/ns")
        print(f"  m_pred        :  {m_pred_best:.5f}")
        print(f"  excess rate   :  {rate_best:.6f}  events/ns²")
        print(f"  δt = t_cut − t0: {delta_t:.1f} ns")
        print(f"{'−'*56}")
        print(f"  err_N (slope) :  ±{err_N_slope:.2f}")
        print(f"  err_N (t0)    :  +{err_N_t0_hi:.2f} / −{err_N_t0_lo:.2f}")
        print(f"  N_sideband    :  {N_best:.1f}  +{err_N_hi:.1f} / −{err_N_lo:.1f}  "
              f"(+{_fhi:.1f}% / −{_flo:.1f}%)")
        print(f"{'*'*56}")
        print(f"  SIGNAL BAND  (× {EXPOSURE_SCALE})")
        print(f"{'*'*56}")
        print(f"  N_signal      :  {EXPOSURE_SCALE*N_best:.1f}  "
              f"+{EXPOSURE_SCALE*err_N_hi:.1f} / −{EXPOSURE_SCALE*err_N_lo:.1f}")
        print(f"{'*'*56}\n")

    return {
        # ── fit parameters ──────────────────────────────────────────────────
        't0_best'       : t0_best,
        't0_err'        : (t0_err_lo, t0_err_hi),
        'C_best'        : C_best,
        'm_data_best'   : m_data_best,
        'm_pred_best'   : m_pred_best,
        'rate_best'     : rate_best,
        # ── goodness of fit ─────────────────────────────────────────────────
        'chi2_min'      : chi2_min,
        'ndof'          : ndof,
        # ── skyshine yield ──────────────────────────────────────────────────
        'N_sideband'    : N_best,
        'N_signal'      : EXPOSURE_SCALE * N_best,
        'err_N'         : (err_N_lo, err_N_hi),
        'err_N_signal'  : (EXPOSURE_SCALE * err_N_lo, EXPOSURE_SCALE * err_N_hi),
        # ── profile arrays ──────────────────────────────────────────────────
        't0_grid'       : t0_grid,
        'chi2_profile'  : chi2_data,
        'N_profile'     : N_arr,
        'in_sigma'      : in_sigma,
    }


# ══════════════════════════════════════════════════════════════════════════════
# 2.  PROFILE DIAGNOSTIC FIGURE
# ══════════════════════════════════════════════════════════════════════════════

def plot_profile_diagnostics(result, t_cut, pot_data, EXPOSURE_SCALE,
                             plot_name=None, SAVE_PLOT=False):
    """
    Two-panel diagnostic figure for the profile likelihood scan.

    Left  — χ²(t0) with the Δχ²=1 band highlighted.
    Right — N_signal(t0) showing how sensitively the result depends on t0;
            the 1-σ band from the profile is shaded in orange.

    Parameters
    ----------
    result         : dict returned by profile_skyshine
    t_cut          : float  signal-window upper edge [ns] (for axis labels)
    pot_data       : float  data POT (×10²⁰) for header annotation
    EXPOSURE_SCALE : float  sideband→signal scaling (for axis labels)
    plot_name      : str | None   base filename; '_profile' is appended before the extension
    SAVE_PLOT      : bool

    Returns
    -------
    matplotlib Figure
    """
    t0_grid     = result['t0_grid']
    chi2_prof   = result['chi2_profile']
    N_prof      = result['N_profile']
    t0_best     = result['t0_best']
    t0_err_lo, t0_err_hi = result['t0_err']
    chi2_min    = result['chi2_min']
    ndof        = result['ndof']
    in_sigma    = result['in_sigma']
    N_sig_best  = result['N_signal']
    err_lo_sig, err_hi_sig = result['err_N_signal']

    N_sig_prof = EXPOSURE_SCALE * N_prof   # scale whole profile to signal band

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))

    # ── left: χ²(t0) profile ─────────────────────────────────────────────────
    ax1.plot(t0_grid, chi2_prof, 'b-', lw=1.5, zorder=3)

    # Δχ² = 1 band
    ax1.axhline(chi2_min + 1.0, color='r', ls='--', lw=1.2,
                label=r'$\Delta\chi^2 = 1$  (1$\sigma$)', zorder=2)
    ax1.fill_between(t0_grid, chi2_min - 0.5, chi2_min + 1.0,
                     where=in_sigma, alpha=0.15, color='red',
                     label='1$\\sigma$ in $t_0$')

    ax1.axvline(t0_best, color='k', ls=':', lw=1.2, zorder=4,
                label=(f'$t_0 = {t0_best:.0f}'
                       f'^{{+{t0_err_hi:.0f}}}_{{-{t0_err_lo:.0f}}}$ ns'))

    ax1.set_xlabel(r'$t_0$ [ns]', fontsize=12)
    ax1.set_ylabel(r'$\chi^2(t_0)$  [data]', fontsize=12)
    ax1.set_ylim([chi2_min - 1.5, chi2_min + 14])
    ax1.set_title(r'Profile likelihood in $t_0$', fontsize=11)
    ax1.legend(fontsize=9, frameon=False)
    #ax1.text(0.03, 1.03, 'ANNIE', fontweight='bold', color='blue',
    #         transform=ax1.transAxes, fontsize=8)
    #ax1.text(0.22, 1.03, 'Preliminary', transform=ax1.transAxes, fontsize=8)
    #ax1.text(0.75, 1.03, f'{pot_data:.2f}' + r' $\times10^{20}$ POT',
    #         transform=ax1.transAxes, fontsize=9)

    # ── right: N_signal vs t0 ────────────────────────────────────────────────
    # filter NaN for clean plot
    valid = np.isfinite(N_sig_prof)

    ax2.plot(t0_grid[valid], N_sig_prof[valid], 'b-', lw=1.5, zorder=3)

    # shade the t0 1-sigma band to show its contribution to σ_N
    ax2.fill_between(t0_grid, N_sig_prof,
                     where=(in_sigma & valid),
                     alpha=0.30, color='orange', zorder=2,
                     label='$t_0$ 1$\\sigma$ band')

    ax2.axvline(t0_best, color='k', ls=':', lw=1.2, zorder=4)
    ax2.axhline(N_sig_best, color='k', ls='--', lw=0.8, alpha=0.5)

    # result annotation
    ax2.text(0.2, 0.3,
             (f'$N_\\mathrm{{sig}} = {N_sig_best:.1f}'
              f'^{{+{err_hi_sig:.1f}}}_{{-{err_lo_sig:.1f}}}$'),
             transform=ax2.transAxes, fontsize=12)
    ax2.text(0.2, 0.22,
             f'(scale = {EXPOSURE_SCALE:.3f})',
             transform=ax2.transAxes, fontsize=9, color='gray')

    ax2.set_xlabel(r'$t_0$ [ns]', fontsize=12)
    ax2.set_ylabel(r'$N_\mathrm{signal}(t_0)$', fontsize=12)
    ax2.set_title(r'Skyshine events vs. assumed $t_0$', fontsize=11)
    ax2.legend(fontsize=9, frameon=False)
    #ax2.text(0.03, 1.03, 'ANNIE', fontweight='bold', color='blue',
    #         transform=ax2.transAxes, fontsize=10)
    #ax2.text(0.22, 1.03, 'Preliminary', transform=ax2.transAxes, fontsize=10)

    plt.tight_layout()
    if SAVE_PLOT and plot_name:
        base, ext = os.path.splitext(plot_name)
        out = f'{base}_profile{ext}'
        plt.savefig(out, dpi=300, bbox_inches='tight')
        print(f"[profile diagnostics] saved → {out}")
    plt.show()
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# 3.  MAIN SIDEBAND COMPARISON PLOT
# ══════════════════════════════════════════════════════════════════════════════

def plot_sideband_comparison(bins, mc_data, MC_weights_by_cat, on_beam_data, off_beam_data,
                             pot_mc, pot_data, trig_ext, trig_data,
                             axis_label, y_axis_label, x_limit, y_limit,
                             xaxis_minor, plot_name,
                             legend_columns=3,
                             DEBUG=False,
                             ALPHA_SCALE=0.18,
                             CALCULATE_SKYSHINE=False,
                             PLOT_SKYSHINE_FIT=False,
                             PLOT_PROFILE=True,
                             SAVE_PLOT=False,
                             t_cut=None,
                             EXPOSURE_SCALE=1.56):
    """
    Stacked MC+data sideband comparison with profile-likelihood skyshine extraction.

    The function
      1. builds the prediction stack and data/prediction histograms,
      2. calls profile_skyshine when CALCULATE_SKYSHINE=True,
      3. draws the main comparison figure  (stacked fill + residuals),
      4. calls plot_profile_diagnostics when PLOT_PROFILE=True.

    Parameters
    ----------
    bins             array   bin edges [ns] (uniform)
    mc_data          list    6 MC arrays [Ext_MC, OOF, CC, NCother, nubarNCQE, nuNCQE]
                             already reordered for the desired stack order
    on_beam_data     array   on-beam bunch-time array
    off_beam_data    array   off-beam (EXT) bunch-time array
    pot_mc           float   MC POT (×10²⁰)
    pot_data         float   data POT (×10²⁰)
    trig_ext         float   total EXT triggers
    trig_data        float   total on-beam triggers
    axis_label       str     x-axis label
    y_axis_label     str     y-axis label
    x_limit          [lo, hi]
    y_limit          [lo, hi]
    xaxis_minor      float   minor tick spacing
    plot_name        str     output filename (used for both figures)
    legend_columns   int     default 3
    DEBUG            bool    verbose stdout
    ALPHA_SCALE      float   dirt-MC normalisation scale (default 0.18)
    CALCULATE_SKYSHINE bool  run profile_skyshine
    PLOT_SKYSHINE_FIT  bool  overlay piecewise fit on main plot
    PLOT_PROFILE     bool    show diagnostic figure (requires CALCULATE_SKYSHINE)
    SAVE_PLOT        bool    save both figures to disk
    t_cut            float | None   signal-window upper edge [ns];
                             defaults to centers_data_23[25] − centers[0]
    EXPOSURE_SCALE   float   sideband → signal scaling (default 1.56)

    Returns
    -------
    dict from profile_skyshine  (None if CALCULATE_SKYSHINE=False)
    """

    # ── 1. Scaling factors ───────────────────────────────────────────────────
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext

    # ── 2. Prediction stack ──────────────────────────────────────────────────
    # Order: [Ext_MC, OOF, CC, NCother, nubarNCQE, nuNCQE]  + Off-Beam Data
    prediction_data = list(mc_data) + [off_beam_data]

    # Per-event GENIE weights * POT scale, falling back to uniform if not provided
    weights = [np.array(MC_weights_by_cat[i]) * f_pot for i in range(len(mc_data))]
    weights.append(np.ones(len(off_beam_data)) * f_ext)

    labels_base   = ['External (MC)', 'Out-of-FV', 'CC', 'NCother',
                     r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors        = ['papayawhip', 'lime', 'darkred', 'cyan',
                     'darkorange', '#0066ff', 'gray']
    color_outline = ['navy'] * 6 + ['black']

    if DEBUG:
        print(f"f_pot scaling factor: {f_pot:.3f}")
        print(f"Dirt scaling factor:  {ALPHA_SCALE}")
        print(f"Raw Dirt events before weights: {len(prediction_data[0])}")
        print(f"Dirt Yield BEFORE scale: {np.sum(weights[0]):.1f}")

    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * ALPHA_SCALE
            if DEBUG:
                print(f"Applied {ALPHA_SCALE} scale to index {i} ({label})")

    if DEBUG:
        print(f"Dirt Yield AFTER scale:  {np.sum(weights[0]):.1f}")
        print(f"Off-Beam Yield: {np.sum(weights[-1]):.1f}")

    # ── 3. Histograms ────────────────────────────────────────────────────────
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths  = np.diff(bins)

    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_counts    = data_counts.astype(float)
    data_err       = np.sqrt(data_counts)
    data_err[data_err == 0] = 1.0

    pred_hist_comps = [
        np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0]
        for i in range(len(prediction_data))
    ]
    pred_counts = np.sum(pred_hist_comps, axis=0).astype(float)

    pred_w2_comps = [
        np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0]
        for i in range(len(prediction_data))
    ]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))
    pred_err[pred_err == 0] = 1.0

    # ── 4. Profile likelihood skyshine ───────────────────────────────────────
    profile_result = None
    if CALCULATE_SKYSHINE:
        if t_cut is None:
            t_cut = centers_data_23[25] - centers_data_23[0]   # fall back to env globals

        profile_result = profile_skyshine(
            bins, data_counts, data_err,
            pred_counts, pred_err,
            t_cut=t_cut,
            EXPOSURE_SCALE=EXPOSURE_SCALE,
            DEBUG=DEBUG,
        )

    # ── 5. Yields for legend labels ──────────────────────────────────────────
    yields           = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1.0
    labels = [
        f"{l} ({100*y/total_pred_yield:.1f}%)"
        for l, y in zip(labels_base, yields)
    ]

    # ── 6. Residuals ─────────────────────────────────────────────────────────
    mask    = pred_counts > 0
    res     = np.full_like(pred_counts, np.nan)
    res_err = np.full_like(pred_counts, np.nan)

    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    ratio      = data_counts[mask] / pred_counts[mask]
    d_term     = np.where(data_counts[mask] > 0,
                          (data_err[mask] / data_counts[mask])**2, 0.0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask] / pred_counts[mask])**2)

    # ── 7. Main figure ───────────────────────────────────────────────────────
    fig = plt.figure(figsize=(7, 7))
    gs  = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax  = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # stacked filled histogram (no edges → clean look)
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights,
            label=labels, color=colors, linewidth=0)
    # category outlines
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights,
            histtype='step', color=color_outline, linewidth=0.8)
    # on-beam data
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2,
                fmt='ko', label='On-Beam Data', markersize=4, capsize=0)

    # profile fit overlay ─────────────────────────────────────────────────────
    if PLOT_SKYSHINE_FIT and profile_result is not None:
        x_fit = np.linspace(x_limit[0], x_limit[1], 1000)
        t0_pl = profile_result['t0_best']
        C_pl  = profile_result['C_best']
        m_pl  = profile_result['m_data_best']
        y_fit = np.where(x_fit < t0_pl,
                         C_pl,
                         C_pl + m_pl * (x_fit - t0_pl))
        ax.plot(x_fit, y_fit, color='magenta', ls='--', lw=2.0, zorder=5,
                label=f'Skyshine fit')

        chi2_txt = (r"$\chi^2/\mathrm{dof} = $"
                    + f"{profile_result['chi2_min']:.1f}/"
                    + f"{profile_result['ndof']}")
        ax.text(0.1, 0.72, chi2_txt, transform=ax.transAxes, fontsize=12)

    # residual panel
    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask],
                 xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline( 0.0, color='gray', ls='--', lw=1.0)
    ax2.axhline( 0.5, color='gray', ls=':',  lw=0.8)
    ax2.axhline(-0.5, color='gray', ls=':',  lw=0.8)

    # formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)
    ax.text(0.02, 1.02, 'ANNIE',      fontweight='bold', color='blue',
            transform=ax.transAxes)
    #ax.text(0.17, 1.02, 'Preliminary', transform=ax.transAxes)
    ax.text(0.75, 1.02, f"{pot_data:.2f}" + r" $\times 10^{20}$ POT",   # nominally 0.70
            transform=ax.transAxes, fontsize = 12)
    
    ax.set_title('Skyshine Sideband', fontsize = 14, weight = 'bold')

    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=16)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 5])
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))

    plt.tight_layout()
    if SAVE_PLOT:
        plt.savefig(plot_name, dpi=300, bbox_inches='tight')
        print(f"[sideband] saved → {plot_name}")
    plt.show()

    # ── 8. Profile diagnostics figure ────────────────────────────────────────
    if CALCULATE_SKYSHINE and PLOT_PROFILE and profile_result is not None:
        plot_profile_diagnostics(
            profile_result,
            t_cut=t_cut,
            pot_data=pot_data,
            EXPOSURE_SCALE=EXPOSURE_SCALE,
            plot_name=plot_name,
            SAVE_PLOT=SAVE_PLOT,
        )

    return profile_result


# ══════════════════════════════════════════════════════════════════════════════
# 4.  DRIVER  (edit as needed)
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    # ── binning ──────────────────────────────────────────────────────────────
    BUNCHES_PER_BIN = 10
    bin_width = 18.936 * BUNCHES_PER_BIN
    bins = np.arange(0, 18.936 * 80 + bin_width, bin_width)

    # ── MC stack (reordered for desired colour order) ─────────────────────────
    mc_stack = [
        bunches_80_mc[5],   # nuNCQE    (bottom)
        bunches_80_mc[4],   # nubarNCQE
        bunches_80_mc[2],   # CC
        bunches_80_mc[1],   # OOF
        bunches_80_mc[3],   # NCother
        bunches_80_mc[0],   # External MC (dirt)   ← scaled by ALPHA_SCALE
    ]
    
    mc_weights_stack = [
        bunches_80_weights[5],
        bunches_80_weights[4],
        bunches_80_weights[2],
        bunches_80_weights[1],
        bunches_80_weights[3],
        bunches_80_weights[0]
    ]

    # ── common scaling arguments ──────────────────────────────────────────────
    common_args = dict(
        pot_mc    = POT_MC,
        pot_data  = POT_DATA,
        trig_ext  = TRIG_EXT,
        trig_data = TRIG_DATA,
    )

    t_cut = centers_data_23[25] - centers_data_23[0]   # 1/3-spill cut for the final signal
    
    print(t_cut)

    # ── call ─────────────────────────────────────────────────────────────────
    result = plot_sideband_comparison(
        bins            = bins,
        mc_data         = mc_stack,
        MC_weights_by_cat=mc_weights_stack,
        on_beam_data    = bunches_80_data,
        off_beam_data   = bunches_80_ext,
        axis_label      = 'Time since first bunch [ns]',
        y_axis_label    = f'Events / {BUNCHES_PER_BIN} BNB bunches',
        x_limit         = [0, 18.936 * 80],
        y_limit         = [0, 500],
        xaxis_minor     = 50,
        plot_name       = '../plots/sideband/post_graduation/1sigma_expanded_sideband/sideband_timing_fit_v2_1sigma.png',
        legend_columns  = 3,
        DEBUG           = True,
        ALPHA_SCALE     = 0.18,
        CALCULATE_SKYSHINE = True,
        PLOT_SKYSHINE_FIT  = True,
        PLOT_PROFILE       = True,
        SAVE_PLOT          = False,
        t_cut           = t_cut,
        EXPOSURE_SCALE  = 0.623,#1.569,     # calculated from the sideband / signal exposure
        **common_args,
    )

    # ── access results downstream ─────────────────────────────────────────────
    if result is not None:
        N  = result['N_signal']
        lo, hi = result['err_N_signal']
        print(f"\nSkyshine estimate (signal band): {N:.1f}  +{hi:.1f} / −{lo:.1f}")

In [ ]:
# simple event rate bin for dirt scaling factor

# ---------------------------------------------------------
# 1D RATE NORMALIZATION (Total Yield Matching)
# ---------------------------------------------------------
def compute_dirt_scale_with_skyshine(
        mc_stack,
        data_events,
        ext_events,
        expected_skyshine_yield,
        pot_mc,
        pot_data,
        trig_ext,
        trig_data,
        mc_weights_stack=None,
        nominal_dirt_scale=0.18,
        verbose=True):

    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext

    def mc_yield(i):
        if mc_weights_stack is not None:
            return float(np.sum(mc_weights_stack[i])) * f_pot
        return len(mc_stack[i]) * f_pot

    total_data = len(data_events)

    # Fixed MC: everything EXCEPT dirt (index 0 = mc_stack[0] = dirt)
    total_fixed_mc  = sum(mc_yield(i) for i in range(1, len(mc_stack)))  # ← range(1,...)
    total_off_beam  = len(ext_events) * f_ext
    total_skyshine  = expected_skyshine_yield
    total_fixed     = total_fixed_mc + total_off_beam + total_skyshine

    # Dirt yield: GENIE-weighted and POT-scaled, but NO 0.18 applied
    total_dirt_unscaled = mc_yield(0)                                     # ← mc_yield, not len()*f_pot

    # Absolute dirt scale: solve alpha * dirt = residual
    if total_dirt_unscaled <= 0:
        absolute_dirt_scale = 0.0
    else:
        absolute_dirt_scale = (total_data - total_fixed) / total_dirt_unscaled

    if verbose:
        residual          = total_data - total_fixed
        nominal_dirt_yield = total_dirt_unscaled * nominal_dirt_scale

        print("\n" + "="*52)
        print("  1D RATE NORMALIZATION (WITH SKYSHINE)")
        print("="*52)
        print(f"  Total Data Events:            {total_data}")
        print(f"  Fixed MC (excl. dirt):        {total_fixed_mc:.1f}")
        print(f"  Off-beam:                     {total_off_beam:.1f}")
        print(f"  Skyshine (fixed):             {total_skyshine:.1f}")
        print(f"  Total Fixed Backgrounds:      {total_fixed:.1f}")
        print(f"  Dirt (GENIE×POT, no α):       {total_dirt_unscaled:.1f}")
        print("-"*52)
        print(f"  Residual (data − fixed):      {residual:.1f}")
        print(f"  Nominal dirt yield (×{nominal_dirt_scale:.2f}):   {nominal_dirt_yield:.1f}")
        print("-"*52)
        print(f"  Nominal ALPHA_SCALE:           {nominal_dirt_scale:.4f}")
        print(f"  Absolute ALPHA_SCALE:          {absolute_dirt_scale:.4f}")
        print(f"  Relative adjustment:           ×{absolute_dirt_scale/nominal_dirt_scale:.3f}  "
              f"({'↑' if absolute_dirt_scale > nominal_dirt_scale else '↓'} from nominal)")
        print("="*52 + "\n")

    return absolute_dirt_scale


mc_stack = [bunches_80_mc[5], bunches_80_mc[4], bunches_80_mc[2], 
            bunches_80_mc[1], bunches_80_mc[3], bunches_80_mc[0]]

mc_weights_stack = [
    bunches_80_weights[5],
    bunches_80_weights[4],
    bunches_80_weights[2],
    bunches_80_weights[1],
    bunches_80_weights[3],
    bunches_80_weights[0],
]

alpha = compute_dirt_scale_with_skyshine(
    mc_stack=mc_stack,
    mc_weights_stack=mc_weights_stack,
    data_events=bunches_80_data,
    ext_events=bunches_80_ext,
    expected_skyshine_yield=10.0,   # corrected value
    pot_mc=POT_MC, pot_data=POT_DATA,
    trig_ext=TRIG_EXT, trig_data=TRIG_DATA,
    nominal_dirt_scale=0.18,
)

In [ ]:
def plot_with_skyshine(bins, mc_data, on_beam_data, off_beam_data, skyshine_data, 
                       pot_mc, pot_data, trig_ext, trig_data, expected_skyshine_yield,
                       axis_label, y_axis_label, x_limit, y_limit, xaxis_minor, 
                       plot_name, legend_columns=3,
                       mc_weights_stack=None, 
                       DEBUG=False, ALPHA_SCALE=0.18, SAVE_PLOT=False):
    
    # 1. Scaling Factors
    f_pot = pot_data / pot_mc
    f_ext = trig_data / trig_ext
    
    ### NEW: Calculate the weight required to force the skyshine MC to exactly match your expected yield
    if len(skyshine_data) > 0:
        f_sky = expected_skyshine_yield / len(skyshine_data)
    else:
        f_sky = 0.0
    
    # 2. Prepare Prediction Stack (MC + Skyshine + Off-Beam)
    ### NEW: We inject Skyshine right after External (MC) [which is index 0]
    prediction_data = [mc_data[0], skyshine_data] + mc_data[1:] + [off_beam_data]
    
    ### NEW: We build the weights array to match the new 8-element prediction_data order
    def mc_w(i):
        if mc_weights_stack is not None:
            return np.array(mc_weights_stack[i]) * f_pot
        return np.ones(len(mc_data[i])) * f_pot

    weights = (
            [mc_w(0),                               # dirt
             np.ones(len(skyshine_data)) * f_sky]   # skyshine — no GENIE weights
          + [mc_w(i) for i in range(1, len(mc_data))]  # remaining MC cats
          + [np.ones(len(off_beam_data)) * f_ext]   # off-beam
        )
    
    # 3. Aesthetics
    ### NEW: Added Skyshine to labels and colors and increased outlines to 7
    labels_base = ['External (MC)', 'Skyshine', 'Out-of-FV', 'CC', 'NCother', r'$\bar{\nu}$NCQE', r'$\nu$NCQE', 'Off-Beam Data']
    colors = ['papayawhip', 'darkmagenta', 'lime', 'darkred', 'cyan', 'darkorange', '#0066ff', 'gray']
    color_outline = ['navy'] * 7 + ['black']
    
    for i, label in enumerate(labels_base):
        if 'External (MC)' in label:
            weights[i] = weights[i] * ALPHA_SCALE
            if DEBUG:
                print(f"Applied {ALPHA_SCALE} scale to index {i} ({label})")
        ### NEW: Just a helpful printout to confirm the math worked
        if 'Skyshine' in label:
            if DEBUG:
                print(f"Scaled {len(skyshine_data)} raw skyshine events to exactly {expected_skyshine_yield:.1f} events")

    # Calculate scaled yields for labels
    yields = [np.sum(w) for w in weights]
    total_pred_yield = sum(yields) if sum(yields) > 0 else 1
    labels = [f"{l} ({100*y/total_pred_yield:.1f}%)" for l, y in zip(labels_base, yields)]
    
    # 4. Math for Residuals and Errors
    bin_centers = 0.5 * (bins[1:] + bins[:-1])
    bin_widths = np.diff(bins)
    
    data_counts, _ = np.histogram(on_beam_data, bins=bins)
    data_err = np.sqrt(data_counts)
    
    pred_hist_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i])[0] for i in range(len(prediction_data))]
    pred_counts = np.sum(pred_hist_comps, axis=0)
    
    pred_w2_comps = [np.histogram(prediction_data[i], bins=bins, weights=weights[i]**2)[0] for i in range(len(prediction_data))]
    pred_err = np.sqrt(np.sum(pred_w2_comps, axis=0))

    # --- Plotting ---
    fig = plt.figure(figsize=(7, 7))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.1)
    ax = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])

    # A. The Fill
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            label=labels, color=colors, linewidth=0)

    # B. The Outline
    ax.hist(prediction_data, bins=bins, stacked=True, weights=weights, 
            histtype='step', color=color_outline, linewidth=0.8)

    # C. On-Beam Data Points
    ax.errorbar(bin_centers, data_counts, yerr=data_err, xerr=bin_widths/2, 
                fmt='ko', label='On-Beam Data', markersize=4, capsize=0)

    # --- Residual Plot: (Data - Prediction) / Prediction ---
    mask = pred_counts > 0
    res = np.full_like(pred_counts, np.nan, dtype=float)
    res_err = np.full_like(pred_counts, np.nan, dtype=float)
    
    res[mask] = (data_counts[mask] - pred_counts[mask]) / pred_counts[mask]
    
    ratio = data_counts[mask] / pred_counts[mask]
    d_term = np.where(data_counts[mask] > 0, (data_err[mask]/data_counts[mask])**2, 0)
    res_err[mask] = ratio * np.sqrt(d_term + (pred_err[mask]/pred_counts[mask])**2)

    ax2.errorbar(bin_centers[mask], res[mask], yerr=res_err[mask], xerr=bin_widths[mask]/2, fmt='ko', markersize=4)
    ax2.axhline(0, color='gray', linestyle='--', linewidth=1)
    ax2.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
    ax2.axhline(-0.5, color='gray', linestyle=':', linewidth=0.8)
    
    # Formatting
    ax.set_ylabel(y_axis_label)
    ax.set_xlim(x_limit)
    ax.set_ylim(y_limit)
    ax.tick_params(axis='x', labelbottom=False)
    ax.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax2.set_ylabel(r'$\frac{\mathrm{Data}}{\mathrm{Pred}} - 1$', fontsize=14)
    ax2.set_xlabel(axis_label)
    ax2.set_xlim(x_limit)
    ax2.set_ylim([-1.2, 1.2]) 
    ax2.xaxis.set_minor_locator(MultipleLocator(xaxis_minor))
    
    ax.legend(ncol=legend_columns, fontsize=8, loc='upper right', frameon=False)
    
    ax.text(0.02, 1.02, 'ANNIE', fontweight='bold', color='blue', transform=ax.transAxes)
    ax.set_title('Skyshine Sideband', weight = 'bold', fontsize = 14)
    #ax.text(0.17, 1.02, 'Preliminary', transform=ax.transAxes)
    ax.text(0.70, 1.02, f"{pot_data:.2f} " + r"$\times 10^{20}$ POT", transform=ax.transAxes, fontsize = 12)

    plt.tight_layout()
    if SAVE_PLOT:
        plt.savefig(plot_name, dpi=300, bbox_inches='tight')
    plt.show()
    
print('done')

In [ ]:
path_folder = '../plots/sideband/post_graduation/observables/'

radius_skyshine = np.sqrt(np.array(skyshine_vertex_x)**2 + np.array(skyshine_vertex_z)**2)

common_args = {
        'pot_mc': POT_MC,
        'pot_data': POT_DATA,
        'trig_ext': TRIG_EXT,
        'trig_data': TRIG_DATA,
        'legend_columns': 3
    }

# Expected skyshine from fit within the sideband region
EXPECTED_SKYSHINE = 10.0

mc_weights_stack = [
    MC_weights_by_cat[5],
    MC_weights_by_cat[4],
    MC_weights_by_cat[2],
    MC_weights_by_cat[1],
    MC_weights_by_cat[3],
    MC_weights_by_cat[0],
]


# energy
mc_stack = [MC_reco_E[5], MC_reco_E[4], MC_reco_E[2], MC_reco_E[1], MC_reco_E[3], MC_reco_E[0]]
plot_with_skyshine(
    bins=np.arange(5, 13, 1),
    mc_data=mc_stack,
    mc_weights_stack=mc_weights_stack,
    on_beam_data=reco_E,
    off_beam_data=ext_reco_E,
    skyshine_data=skyshine_reco_E, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed energy [MeV]',
    y_axis_label='Events / 1 MeV bin',
    x_limit=[5, 12],
    y_limit=[0,80],
    xaxis_minor=0.5,
    plot_name=path_folder + 'skyshine_sideband_reco_energy.png',
    DEBUG=True,
    SAVE_PLOT=True,
    **common_args
)


mc_stack_hits = [MC_hits[5], MC_hits[4], MC_hits[2], MC_hits[1], MC_hits[3], MC_hits[0]]
plot_with_skyshine(
    bins=np.arange(10, 50, 4),
    mc_data=mc_stack_hits,
    mc_weights_stack=mc_weights_stack,
    on_beam_data=hits,
    off_beam_data=ext_hits,
    skyshine_data=skyshine_hits, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='PMT hits',
    y_axis_label='Events / 4 hit bin',
    x_limit=[10, 50],
    y_limit=[0,100],
    xaxis_minor=1,
    plot_name=path_folder + 'skyshine_sideband_PMT_hits.png',
    DEBUG=True,
    SAVE_PLOT=True,
    **common_args
)

mc_stack_r = [radius_mc[5], radius_mc[4], radius_mc[2], radius_mc[1], radius_mc[3], radius_mc[0]]
plot_with_skyshine(
    bins=np.arange(0, 0.8, 0.1),
    mc_data=mc_stack_r,
    mc_weights_stack=mc_weights_stack,
    on_beam_data=radius_data,
    off_beam_data=radius_ext,
    skyshine_data=radius_skyshine, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed distance to center [m]',
    y_axis_label='Events / 10 cm bin',
    x_limit=[0, 0.7],
    y_limit=[0,100],
    xaxis_minor=0.1,
    plot_name=path_folder + 'skyshine_sideband_radius.png',
    DEBUG=True,
    SAVE_PLOT=True,
    **common_args
)

mc_stack_y = [MC_vertex_y[5], MC_vertex_y[4], MC_vertex_y[2], MC_vertex_y[1], MC_vertex_y[3], MC_vertex_y[0]]
plot_with_skyshine(
    bins=np.arange(-1, 1.25, 0.25),
    mc_data=mc_stack_y,
    mc_weights_stack=mc_weights_stack,
    on_beam_data=vertex_y,
    off_beam_data=ext_vertex_y,
    skyshine_data=skyshine_vertex_y, # <-- Pass the new skyshine simulated energies here
    expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
    axis_label='Reconstructed vertex (Y) [m]',
    y_axis_label='Events / 50 cm bin',
    x_limit=[-1, 1],
    y_limit=[0,100],
    xaxis_minor=0.2,
    plot_name=path_folder + 'skyshine_sideband_Y.png',
    DEBUG=True,
    SAVE_PLOT=True,
    **common_args
)

vars_cv = [('x', MC_CV_x, CV_x, ext_CV_x, skyshine_CV_x),
           ('y', MC_CV_y, CV_y, ext_CV_y, skyshine_CV_y),
           ('z', MC_CV_z, CV_z, ext_CV_z, skyshine_CV_z)]
for name, mc_list, data_list, ext_list, sky_list in vars_cv:
    mc_stack = [mc_list[5], mc_list[4], mc_list[2], mc_list[1], mc_list[3], mc_list[0]]
    plot_with_skyshine(
        bins=np.arange(-1, 1.25, 0.25) if name != 'z' else np.arange(-1, 0.6, 0.2),
        mc_data=mc_stack,
        mc_weights_stack=mc_weights_stack,
        on_beam_data=data_list,
        off_beam_data=ext_list,
        skyshine_data=sky_list, # <-- Pass the new skyshine simulated energies here
        expected_skyshine_yield=EXPECTED_SKYSHINE, # <-- Tell it the target area
        axis_label=r'$C_V$ ' + f'({name})',
        y_axis_label='Events / bin',
        x_limit=[-1, 1] if name != 'z' else [-1, 0.4],
        y_limit=[0,80] if name != 'y' else [0, 100] ,
        xaxis_minor=0.1,
        plot_name=f'{path_folder}skyshine_sideband_Cv_{name}.png',
        DEBUG=True,
        SAVE_PLOT=True,
        **common_args
    )
    

print('done')